# Exp 8 v8: valid-capacity test (d=12 joint) + probe transfer with an untrained baseline

Needs `exp8_lib.py` in the working directory (not rebuilt here; I have not seen it). Everything else is in this notebook.

**Why this notebook exists.** At d=32 and d=16 the joint gate has no headroom (UNTESTABLE). At d=8 there is headroom but the translators cannot even copy (identity heldout_acc 0.65), so a(g) is meaningless. This notebook looks for a capacity where both halves work, and adds one measure of sharing that does not rely on loss gaps.

**Run order. One phase at a time. Every launch cell is guarded by a flag in STEP 2.**

| Phase | Steps | What | Counts as support for the theory? |
|---|---|---|---|
| A | 4a, 4b, 4c | launch: d=12 translators, d=16 translators trained 3x longer, finish d=16 shared joint checkpoints | no, these only check the instruments |
| B | 5a, 5b | read: is the cost measure valid at d=12? do the d=16 costs survive longer training? | no ("the ruler works") |
| C | 6a, 6b | launch d=12 joint first pass (only if 5a passed), then read gate, per-rendering table, Check 1 | **yes** (Check 1 is the non-obvious claim; needs seeds 7, 13) |
| D | 7a, 7b, 7c | probe transfer on d=16 shared checkpoints vs an untrained baseline vs a(g) | **yes**, weaker (n<=10, one seed) |
| E | 8 | seeds 7 and 13 at d=12, Check 2 | needed before any verdict |

Gate and checks refuse to print until every expected file exists. A failed gate means untestable, not negative.

## STEP 1a. Definitions: checks 1 and 2 (unchanged from v7, CPU only)

In [2]:
# Definitions for checks 1 and 2 (CPU only). Nothing runs here.
import os, sys, json, glob, random, time, subprocess
import numpy as np
import exp8_lib as X

# ----------------------------------------------------------------------------- frozen thresholds (edit only BEFORE joint results exist)
GATE_MIN_GAIN, GATE_Z = 0.02, 3.0        # same instrument gate as the notebook
C1_MARGIN        = 0.10                  # s(rev_all) - max(s(rev4), s(rev45)) must be >= this in EVERY seed
C1_MIN_SEEDS     = 3                     # fewer seeds -> no verdict
C2_MIN_N         = 8                     # renderings with finite a and s
C2_MIN_SEEDS     = 3                     # fewer seeds -> numbers are printed but there is no verdict
C2_PARTIAL_PASS  = -0.5                  # partial rho(a, s | tau, displacement) must be <= this
C2_PARTIAL_DEAD  = -0.2                  # ... and is DEAD if >= this
C2_LOO_MAX       = -0.3                  # leave-one-rendering-out partial must never exceed this
C2_BEAT_MARGIN   = 0.10                  # rho(a, s) must be at least this much more negative than rho(tau, s)

PREREG = f"""H-adapter pre-registration - written {{when}}
Written before ANY joint (s) result was read. Translator costs a(g) for seed 0 had already been seen.

GATE   mean gain_A' >= {GATE_MIN_GAIN} nats and >= {GATE_Z} SE. If it fails, every check below is UNTESTABLE, not negative.

CHECK 1 (dissociation) needs >= {C1_MIN_SEEDS} seeds.
  PASS         s(rev_all) - max(s(rev4), s(rev45)) >= {C1_MARGIN} in every seed
  FAIL         mean of that difference <= 0
  INCONCLUSIVE otherwise
  Secondary (reported, not part of the verdict): perm_sym is among the two lowest s in every seed.

CHECK 2 (a beats string distance) needs >= {C2_MIN_N} renderings with finite a and s, and >= {C2_MIN_SEEDS} seeds.
  PASS         partial rho(a, s | tau, displacement) <= {C2_PARTIAL_PASS}
               AND rho(a, s) <= rho(tau, s) - {C2_BEAT_MARGIN}
               AND the leave-one-rendering-out partial never exceeds {C2_LOO_MAX}
  FAIL         partial >= {C2_PARTIAL_DEAD}
  INCONCLUSIVE otherwise

CHECK 3 (depth) - see exp8_depth.py for its frozen thresholds.
"""

def write_prereg(root):
    path = os.path.join(root, "prereg_three_checks.txt")
    if os.path.exists(path): print("already frozen:", path); print(open(path).read()); return
    os.makedirs(root, exist_ok=True); open(path, "w").write(PREREG.format(when=time.strftime("%Y-%m-%d %H:%M"))); print("wrote", path)

AMENDMENT_1 = """AMENDMENT 1 ({when}). Found because the gate printed gain_A' = -0.0435 nats for every rendering (z about -30).
Cause: the disjoint control uses permuted grammars, whose per-token entropy floors differ from the shared grammar's (A' floor 0.7379 vs 0.7714 nats/token), so raw CE differences mix grammar entropy with sharing.
Change: every gain is now computed on excess CE over the analytic entropy floor, excess = CE - floor, on the same evaluation trees. Thresholds are unchanged.
This was decided from the gate (anchor language A') only, before any rendering's s was read."""

def amend_prereg(root, text=None):
    path = os.path.join(root, "prereg_three_checks.txt"); text = (text or AMENDMENT_1).format(when=time.strftime("%Y-%m-%d %H:%M"))
    if not os.path.exists(path): print("freeze first: run write_prereg"); return
    if text.split("(")[0] in open(path).read(): print("amendment already recorded"); return
    open(path, "a").write("\n" + text + "\n"); print("amendment appended to", path)

# ----------------------------------------------------------------------------- loading (same definitions as the notebook)
def load_all(root):
    J, T = {}, {}
    for p in glob.glob(os.path.join(root, "joint", "*.json")):
        r = json.load(open(p)); J[(r["g"], r["cond"], r["d"], r["seed"])] = r
    for p in glob.glob(os.path.join(root, "translator", "*.json")):
        r = json.load(open(p)); T[(r["g"], r["d"], r["seed"])] = r
    return J, T

_FLOOR = {}
def _nll(rules, node):
    if isinstance(node, int): return 0.0
    sym, ch = node
    return float(np.log(len(rules[sym]))) + sum(_nll(rules, c) for c in ch)

def floor_ce(cond, lang, g, n=1000, seed=424242, max_leaves=160):
    """Entropy floor (nats per token) of language `lang` in condition `cond`, on exactly the trees the joint evaluation uses (seed 424242, <= 160 leaves).
    Every rendering is injective, so the tokens carry the tree's full information and the best possible per-token CE is nll(tree) / len(rendering).
    The permuted (disjoint) grammars have DIFFERENT entropies from the shared grammar, so raw CE cannot be compared across conditions."""
    key = (cond, lang, g if lang == "G" else "-")
    if key not in _FLOOR:
        gram = X.make_grammars(cond)[lang]; spec = X.TRANSFORMS[g] if lang == "G" else {}; rng = random.Random(seed); v = []
        while len(v) < n:
            t = X.sample_tree(gram, rng)
            if len(X.render(t, {})) > max_leaves: continue
            v.append(_nll(gram, t) / len(X.render(t, spec)))
        _FLOOR[key] = float(np.mean(v))
    return _FLOOR[key]

def gains(J, d, seeds, use_floor=True):
    """gain_L = excess_disjoint(L) - excess_shared(L), with excess = CE - entropy floor. use_floor=False reproduces the raw CE difference (confounded by grammar entropy)."""
    out = {}
    for g in X.G_LIST:
        for s in seeds:
            a, b = J.get((g, "shared", d, s)), J.get((g, "disjoint", d, s))
            if a is None or b is None: continue
            fl = (lambda cond, L: floor_ce(cond, L, g)) if use_floor else (lambda cond, L: 0.0)
            ex_sh = {L: a["ce"][L] - fl("shared", L) for L in X.LANGS3}; ex_dj = {L: b["ce"][L] - fl("disjoint", L) for L in X.LANGS3}
            gain = {L: ex_dj[L] - ex_sh[L] for L in X.LANGS3}
            se = {L: (a["ce_se"][L] ** 2 + b["ce_se"][L] ** 2) ** 0.5 for L in X.LANGS3}
            ok = gain["A2"] > max(GATE_MIN_GAIN, GATE_Z * se["A2"])
            out[(g, s)] = dict(gain=gain, se=se, anchor_ok=ok, s=gain["G"] / gain["A2"] if ok else float("nan"), excess_shared=ex_sh, excess_disjoint=ex_dj)
    return out

def cost_a(T, d, seeds):
    out = {}
    for g in X.G_LIST:
        v = [T[(g, d, s)]["auc"] - T[("identity", d, s)]["auc"] for s in seeds if (g, d, s) in T and ("identity", d, s) in T]
        if v: out[g] = float(np.mean(v))
    return out

def gate_status(G):
    if not G: return False, "no joint runs yet"
    anchors = [v["gain"]["A2"] for v in G.values()]; ses = [v["se"]["A2"] for v in G.values()]
    m = float(np.mean(anchors)); sem = max(float(np.mean(ses)) / len(anchors) ** 0.5, 1e-9)
    ok = m >= GATE_MIN_GAIN and m >= GATE_Z * sem
    return ok, f"mean gain_A' {m:.4f} nats over {len(anchors)} runs (SE of mean ~{sem:.4f})"

# ----------------------------------------------------------------------------- stats helpers
def _ranks(x):
    x = np.asarray(x, float); order = np.argsort(x); r = np.empty(len(x)); i = 0
    while i < len(x):
        j = i
        while j + 1 < len(x) and x[order[j + 1]] == x[order[i]]: j += 1
        r[order[i:j + 1]] = (i + j) / 2 + 1; i = j + 1
    return r

def spearman(x, y):
    rx, ry = _ranks(x), _ranks(y)
    if rx.std() < 1e-12 or ry.std() < 1e-12: return float("nan")
    return float(np.corrcoef(rx, ry)[0, 1])

def _resid(v, Zr): return v - Zr @ np.linalg.lstsq(Zr, v, rcond=None)[0]

def partial_rank_corr(x, y, Z):
    """Spearman of x and y after regressing both ranks on the ranks of every column in Z (Z: list of vectors)."""
    n = len(x); Zr = np.column_stack([np.ones(n)] + [_ranks(z) for z in Z])
    ex, ey = _resid(_ranks(x), Zr), _resid(_ranks(y), Zr)
    if ex.std() < 1e-12 or ey.std() < 1e-12: return float("nan")
    return float(np.corrcoef(ex, ey)[0, 1])

def perm_p_partial(x, y, Z, obs, n=20000, seed=0):
    """one-sided permutation p for partial <= obs, permuting the residual of y (approximate; n is tiny, read it as a rough guide)."""
    rng = np.random.RandomState(seed); m = len(x); Zr = np.column_stack([np.ones(m)] + [_ranks(z) for z in Z])
    ex, ey = _resid(_ranks(x), Zr), _resid(_ranks(y), Zr)
    return float(np.mean([np.corrcoef(ex, rng.permutation(ey))[0, 1] <= obs for _ in range(n)]))

# ----------------------------------------------------------------------------- string baselines (same as the notebook cell)
def _uniq(node, c):
    if isinstance(node, int): c[0] += 1; return 1000 + c[0] - 1
    return (node[0], [_uniq(ch, c) for ch in node[1]])

def string_baselines(g, n_trees=300, seed=3):
    rules, _ = X.build_grammar(X.GRAMMAR_SEED); rng = random.Random(seed); spec = X.TRANSFORMS[g]; disp, tau = [], []
    for _ in range(n_trees):
        t = _uniq(X.sample_tree(rules, rng), [0]); out = X.render(t, spec)
        leaves = np.array([x - 1000 for x in out if x >= 1000]); n = len(leaves); pos = np.empty(n, int); pos[leaves] = np.arange(n)
        disp.append(np.abs(pos - np.arange(n)).mean() / n)
        tau.append(((pos[:, None] > pos[None, :]) & (np.arange(n)[:, None] < np.arange(n)[None, :])).sum() / (n * (n - 1) / 2))
    return dict(displacement=float(np.mean(disp)), kendall_tau=float(np.mean(tau)))

# ----------------------------------------------------------------------------- CHECK 1
def check1(root, d=32, seeds=(0, 7, 13), verbose=True):
    J, T = load_all(root); G = gains(J, d, list(seeds)); ok, msg = gate_status(G)
    print(f"=== CHECK 1: rev_all vs rev4 / rev45  (d={d}) ===\nGATE: {msg} -> {'PASS' if ok else 'FAIL'}")
    if not ok: print("VERDICT: UNTESTABLE (gate). Not a negative result. Try smaller d."); return dict(verdict="UNTESTABLE")
    need = ("rev_all", "rev4", "rev45"); diffs = []; sec = []
    print(f"\n{'seed':>5s} {'s(rev_all)':>10s} {'s(rev4)':>8s} {'s(rev45)':>9s} {'diff':>7s}   perm_sym s / rank among all")
    for s in seeds:
        v = {g: G[(g, s)]["s"] for g in X.G_LIST if (g, s) in G}
        if not all(g in v and np.isfinite(v[g]) for g in need): print(f"{s:>5d}   missing or gated-out runs"); continue
        diff = v["rev_all"] - max(v["rev4"], v["rev45"]); diffs.append(diff)
        fin = sorted([(x, g) for g, x in v.items() if np.isfinite(x)]); low2 = [g for _, g in fin[:2]]
        sec.append("perm_sym" in low2)
        print(f"{s:>5d} {v['rev_all']:10.3f} {v['rev4']:8.3f} {v['rev45']:9.3f} {diff:+7.3f}   {v.get('perm_sym', float('nan')):.3f} / {[g for _, g in fin].index('perm_sym') + 1 if 'perm_sym' in v and np.isfinite(v['perm_sym']) else '-'} of {len(fin)}")
    if len(diffs) < C1_MIN_SEEDS:
        print(f"\nVERDICT: NO VERDICT YET - {len(diffs)} usable seed(s), need {C1_MIN_SEEDS}. (Diffs so far: {[round(x, 3) for x in diffs]}.)"); return dict(verdict="NO VERDICT", diffs=diffs)
    if all(x >= C1_MARGIN for x in diffs): v = "PASS"
    elif float(np.mean(diffs)) <= 0: v = "FAIL"
    else: v = "INCONCLUSIVE"
    print(f"\nmean diff {np.mean(diffs):+.3f}, min {min(diffs):+.3f}, max {max(diffs):+.3f}   |   perm_sym among two lowest in {sum(sec)}/{len(sec)} seeds (secondary)")
    print(f"VERDICT: {v}" + ("  <- the dissociation holds; a(g) is not just string distance" if v == "PASS" else "  <- rev_all does NOT share better; a(g) loses its main argument" if v == "FAIL" else ""))
    return dict(verdict=v, diffs=diffs)

# ----------------------------------------------------------------------------- CHECK 2
def check2(root, d=32, seeds=(0, 7, 13), n_boot=2000):
    J, T = load_all(root); G = gains(J, d, list(seeds)); A = cost_a(T, d, list(seeds)); ok, msg = gate_status(G)
    print(f"=== CHECK 2: does a(g) beat string distance?  (d={d}, seeds={list(seeds)}) ===\nGATE: {msg} -> {'PASS' if ok else 'FAIL'}")
    if not ok: print("VERDICT: UNTESTABLE (gate)."); return dict(verdict="UNTESTABLE")
    gs, a, s = [], [], []
    for g in X.G_LIST:
        v = [G[(g, sd)]["s"] for sd in seeds if (g, sd) in G]; v = [x for x in v if np.isfinite(x)]
        if g in A and v: gs.append(g); a.append(A[g]); s.append(float(np.mean(v)))
    if len(gs) < C2_MIN_N: print(f"VERDICT: NO VERDICT YET - {len(gs)} renderings with finite a and s, need {C2_MIN_N}."); return dict(verdict="NO VERDICT")
    B = {g: string_baselines(g) for g in gs}; tau = [B[g]["kendall_tau"] for g in gs]; disp = [B[g]["displacement"] for g in gs]
    r_as, r_ts, r_ds = spearman(a, s), spearman(tau, s), spearman(disp, s)
    p_both = partial_rank_corr(a, s, [tau, disp]); p_tau = partial_rank_corr(a, s, [tau]); p_disp = partial_rank_corr(a, s, [disp])
    loo = [partial_rank_corr([x for j, x in enumerate(a) if j != i], [x for j, x in enumerate(s) if j != i], [[x for j, x in enumerate(z) if j != i] for z in (tau, disp)]) for i in range(len(gs))]
    rng = np.random.RandomState(1); boots = []
    for _ in range(n_boot):
        ix = rng.randint(0, len(gs), len(gs)); b = partial_rank_corr(np.array(a)[ix], np.array(s)[ix], [np.array(tau)[ix], np.array(disp)[ix]])
        if np.isfinite(b): boots.append(b)
    lo, hi = (np.percentile(boots, [2.5, 97.5]) if boots else (float("nan"),) * 2)
    print(f"\nn = {len(gs)} renderings: {gs}")
    print(f"rho(a, s)               = {r_as:+.2f}")
    print(f"rho(tau, s)             = {r_ts:+.2f}     rho(displacement, s) = {r_ds:+.2f}   <- the baselines")
    print(f"partial rho(a,s | tau)  = {p_tau:+.2f}     | disp = {p_disp:+.2f}     | tau AND disp = {p_both:+.2f}   (bootstrap 95% CI over renderings [{lo:+.2f}, {hi:+.2f}])")
    print(f"leave-one-out partial   range [{np.nanmin(loo):+.2f}, {np.nanmax(loo):+.2f}]   perm p (partial <= observed) = {perm_p_partial(a, s, [tau, disp], p_both):.3f}")
    beats = r_as <= r_ts - C2_BEAT_MARGIN
    n_seeds = len({sd for (_, sd) in G})
    if n_seeds < C2_MIN_SEEDS: print(f"\nVERDICT: NO VERDICT YET - {n_seeds} seed(s) of joint results, need {C2_MIN_SEEDS}. The numbers above are provisional."); return dict(verdict="NO VERDICT", rho_as=r_as, partial=p_both)
    if p_both <= C2_PARTIAL_PASS and beats and np.nanmax(loo) <= C2_LOO_MAX: v = "PASS"
    elif p_both >= C2_PARTIAL_DEAD: v = "FAIL"
    else: v = "INCONCLUSIVE"
    print(f"a beats tau as a predictor of s by >= {C2_BEAT_MARGIN}: {beats}")
    print(f"VERDICT: {v}" + ("  <- a(g) carries information string distance does not" if v == "PASS" else "  <- once tau/displacement are controlled, a(g) predicts nothing" if v == "FAIL" else "  <- more renderings (rev3, rev345, rev2, rev23) and seeds before concluding"))
    print("Note: with ~10 renderings the partial has ~7 degrees of freedom; the bootstrap CI above is wide on purpose. Report it.")
    return dict(verdict=v, rho_as=r_as, rho_ts=r_ts, partial=p_both, ci=(float(lo), float(hi)))

## STEP 1b. Exact entropy floor (unchanged from v7; overrides `floor_ce`)

In [3]:
import math
# Exact entropy floor, -log P(string) per token (CPU only). Replaces floor_ce from STEP 1a.
# Why: the model sees only the string, and a string can have several derivations, so the old floor nll(tree)/tokens was too high
# (by 0.0069 for the shared grammar and 0.0118 for the disjoint A' grammar). Each symbol c gets a matrix M_c[i, j] = P(c yields x[i:j]) in the
# RENDERED order; a rule is the product of its children's matrices (the matrix product sums over split points).
def _order(lvl, idx, n, spec):
    order = list(range(n))
    if lvl in spec.get("rev", ()): order = order[::-1]
    elif lvl in spec.get("swap", ()) and n >= 2: order[0], order[1] = order[1], order[0]
    elif "perm_sym" in spec: order = spec["perm_sym"][((lvl, idx), n)]
    return order

def string_logp(x, rules, sbl, spec):
    x = list(x); n = len(x); L = X.LEVELS; tm = spec.get("term_map", {}); mk = spec.get("marker"); M = {}
    for sym in sbl[L]:
        m = np.zeros((n + 1, n + 1)); deg = len(rules[sym])
        for rhs in rules[sym]:
            order = _order(L, sym[1], len(rhs), spec)
            r = [tm.get(rhs[j], rhs[j]) for j in order] + ([mk] if mk is not None else []); k = len(r)
            for i in range(n - k + 1):
                if x[i:i + k] == r: m[i, i + k] += 1.0 / deg
        M[sym] = m
    for lvl in range(L - 1, 0, -1):
        new = {}
        for sym in sbl[lvl]:
            deg = len(rules[sym]); acc = np.zeros((n + 1, n + 1))
            for rhs in rules[sym]:
                order = _order(lvl, sym[1], len(rhs), spec); P = M[rhs[order[0]]]
                for j in order[1:]: P = P @ M[rhs[j]]
                acc += P / deg
            new[sym] = acc
        M = new
    p = M[(1, 0)][0, n]
    return math.log(p) if p > 0 else float("-inf")

def eval_trees(rules, n_trees=1000, seed=424242, max_leaves=160):      # same trees as X.make_eval_rows
    rng = random.Random(seed); out = []
    while len(out) < n_trees:
        t = X.sample_tree(rules, rng)
        if len(X.render(t, {})) <= max_leaves: out.append(t)
    return out

_FLOOR.clear(); _SBL = X.build_grammar(X.GRAMMAR_SEED)[1]
import multiprocessing as mp
N_PROC = 1          # 1 = safe default (forking inside a Jupyter kernel can hang); raise to e.g. 16 once you have confirmed it does not hang for you
def _chunk(args):
    trees, gram, spec = args
    return [-string_logp(X.render(t, spec), gram, _SBL, spec) / len(X.render(t, spec)) for t in trees]
def floor_ce(cond, lang, g, n=1000, seed=424242, max_leaves=160):
    """Exact floor (nats per token) for language `lang` in condition `cond`; for lang G the rendering g is applied. Cached."""
    same = "canon" if (cond == "shared" and lang in ("A", "A2")) else lang          # shared A and A' are the same grammar, same trees, same rendering
    key = (cond, same, g if lang == "G" else "-")
    if key not in _FLOOR:
        gram = X.make_grammars(cond)[lang]; spec = X.TRANSFORMS[g] if lang == "G" else {}; trees = eval_trees(gram, n, seed, max_leaves)
        if N_PROC > 1:
            with mp.get_context("fork").Pool(N_PROC) as pool: parts = pool.map(_chunk, [(trees[i::N_PROC], gram, spec) for i in range(N_PROC)])
        else: parts = [_chunk((trees, gram, spec))]
        _FLOOR[key] = float(np.mean([v for q in parts for v in q]))
    return _FLOOR[key]
print(f"exact floor_ce installed (replaces the tree-entropy one), {N_PROC} processes. 24 floors are needed for STEP 5 (about 1-2 s each with many cores, ~20 s each with one).")

exact floor_ce installed (replaces the tree-entropy one), 1 processes. 24 floors are needed for STEP 5 (about 1-2 s each with many cores, ~20 s each with one).


## STEP 1c. Definitions: probe transfer / depth (unchanged from v7)

In [4]:
"""exp8_depth.py - Check 3: does cross-rendering sharing start LATER in depth as conversion cost a(g) rises?

Readout (a cheap proxy for the Exp 7 patching; it is a probe, not a causal intervention):
  every token of a rendering has a ground-truth ancestor symbol at each tree level, whatever order the tokens are written in.
  Train a linear probe for the level-k ancestor on canonical-language (A) residual states at layer l, apply it unchanged to rendering-g states.
  T_l(g) = (acc[A-probe on g] - chance) / (acc[g-probe on g] - chance)
  The denominator is g's OWN ceiling (probe trained on g), not A's: reordered strings put different information in each prefix, so A's accuracy is not g's ceiling.
  onset(g) = first layer from which T stays >= ONSET_FRAC to the last layer; NaN if never.

Frozen prediction (edit only before the first run):
  Primary: PRIMARY_LEVEL = 4, centered probes, median over seeds.
  PASS  Spearman(a, onset) >= +0.7 over renderings with a defined onset, AND partial given tau >= +0.5
  FAIL  Spearman(a, onset) <= +0.3
  INCONCLUSIVE otherwise.  UNTESTABLE if fewer than 6 renderings reach ONSET_FRAC, or if the shared-model gate has not passed.
  Reported but not part of the verdict: other levels, strict (uncentered) probes, mean T over layers, an untrained-model floor.
"""
import os, sys, json, random, argparse
import numpy as np
import exp8_lib as X

LEVELS = X.LEVELS
PROBE_LEVELS = [2, 3, 4, 5, 6]          # level 1 has a single symbol, so nothing to decode
PRIMARY_LEVEL = 4
ONSET_FRAC = 0.5
MIN_CEIL_GAP = 0.05                     # layers where the g-probe barely beats chance on g are masked (T undefined)
N_TRAIN, N_TEST = 400, 400              # trees per language per split
TRAIN_SEED, TEST_SEED = 31, 32
C3_PASS_RHO, C3_PASS_PARTIAL, C3_FAIL_RHO, C3_MIN_DEFINED = 0.7, 0.5, 0.3, 6

# ----------------------------------------------------------------------------- rows with ancestor labels (torch-free)
def render_lab(node, spec, anc=()):
    """Mirror of X.render that also returns, per output token, the ancestor symbol index at levels 1..LEVELS.
    A terminal's ancestors are its LEVELS enclosing nonterminals; the marker token belongs to the level-LEVELS node that emitted it."""
    if isinstance(node, int): return [spec.get("term_map", {}).get(node, node)], [anc]
    (lvl, idx), children = node; anc2 = anc + (idx,)
    order = list(range(len(children)))
    if lvl in spec.get("rev", ()): order = order[::-1]
    elif lvl in spec.get("swap", ()) and len(order) >= 2: order[0], order[1] = order[1], order[0]
    elif "perm_sym" in spec: order = spec["perm_sym"][((lvl, idx), len(children))]
    toks, labs = [], []
    for j in order:
        t, l = render_lab(children[j], spec, anc2); toks.extend(t); labs.extend(l)
    if spec.get("marker") is not None and lvl == LEVELS: toks.append(spec["marker"]); labs.append(anc2)
    return toks, labs

def make_rows_lab(grammar, spec, bos, n_trees, seed, max_leaves=160, row_len=X.SEQ_LEN):
    """Same tree sampling as X.make_eval_rows (so the same seed gives the same trees in every language), plus labels [n, row_len, LEVELS] (-1 = no label)."""
    rng = random.Random(seed); rows = np.zeros((n_trees, row_len), np.int16); nreal = np.zeros(n_trees, np.int64); labs = -np.ones((n_trees, row_len, LEVELS), np.int8); k = 0
    while k < n_trees:
        t = X.sample_tree(grammar, rng)
        if len(X.render(t, {})) > max_leaves: continue
        toks, lab = render_lab(t, spec); assert toks == X.render(t, spec), "render_lab diverged from X.render"
        row = [bos] + toks; assert len(row) <= row_len
        rows[k, :len(row)] = row; nreal[k] = len(row); labs[k, 1:len(row)] = np.asarray(lab, np.int8); k += 1
    return rows, nreal, labs

# ----------------------------------------------------------------------------- probes (numpy)
def _ridge_fit(Xtr, y, ncls, lam=1e-2):
    Xb = np.concatenate([Xtr, np.ones((len(Xtr), 1))], 1); Y = np.eye(ncls)[y]
    return np.linalg.solve(Xb.T @ Xb + lam * len(Xb) * np.eye(Xb.shape[1]), Xb.T @ Y)

def _ridge_acc(W, Xte, y): return float((np.argmax(np.concatenate([Xte, np.ones((len(Xte), 1))], 1) @ W, 1) == y).mean())

def probe_transfer(S, Y, ncls=4, centered=True):
    """S[lang][split]: array [L, N, d]; Y[lang][split]: int array [N] (languages 'A', 'A2', 'G'; splits 'train', 'test').
    Returns per-layer dict: acc_AA, acc_AA2, acc_AG, acc_GG, chance, T (NaN where g's own ceiling is within MIN_CEIL_GAP of chance)."""
    L = S["A"]["train"].shape[0]; out = {k: [] for k in ("acc_AA", "acc_AA2", "acc_AG", "acc_GG", "chance", "T")}
    chance = float(np.bincount(Y["G"]["test"], minlength=ncls).max() / len(Y["G"]["test"]))
    for l in range(L):
        mu = {lg: S[lg]["train"][l].mean(0) for lg in S} if centered else {lg: S["A"]["train"][l].mean(0) for lg in S}
        f = lambda lg, sp: S[lg][sp][l] - mu[lg]
        WA = _ridge_fit(f("A", "train"), Y["A"]["train"], ncls); WG = _ridge_fit(f("G", "train"), Y["G"]["train"], ncls)
        aa, aa2 = _ridge_acc(WA, f("A", "test"), Y["A"]["test"]), _ridge_acc(WA, f("A2", "test"), Y["A2"]["test"])
        ag, gg = _ridge_acc(WA, f("G", "test"), Y["G"]["test"]), _ridge_acc(WG, f("G", "test"), Y["G"]["test"])
        T = (ag - chance) / (gg - chance) if gg - chance >= MIN_CEIL_GAP else float("nan")
        for k, v in zip(out, (aa, aa2, ag, gg, chance, T)): out[k].append(float(v))
    return out

def onset_layer(T, frac=ONSET_FRAC):
    """start of the final unbroken run of layers with T >= frac that reaches the last layer. A layer where T is undefined (g's own probe cannot decode yet)
    breaks the run: 'cannot be measured' is not evidence of sharing. NaN if the last layer is below frac or undefined."""
    T = np.asarray(T, float)
    if not np.isfinite(T[-1]) or T[-1] < frac: return float("nan")
    l = len(T) - 1
    while l - 1 >= 0 and np.isfinite(T[l - 1]) and T[l - 1] >= frac: l -= 1
    return float(l)

def decodable_layer(acc_GG, chance, gap=MIN_CEIL_GAP):
    """first layer where g's OWN probe beats chance by `gap`; onset cannot precede it, so onset - decodable is the lag attributable to transfer"""
    a = np.asarray(acc_GG, float) - chance; idx = np.where(a >= gap)[0]
    return float(idx[0]) if len(idx) else float("nan")

# ----------------------------------------------------------------------------- torch part: hidden states at every layer
try:
    import torch
    TORCH = True
except Exception:
    TORCH = False

if TORCH:
    @torch.no_grad()
    def layer_states(model, rows, nreal, labs, dev, bs=64):
        """returns states [L+1, Ntok, d] (embedding output, then every block output) at the rendering tokens (positions 1..nreal-1) and labels [Ntok, LEVELS]"""
        model.eval(); store = []; hs = [model.tok_emb.register_forward_hook(lambda m, i, o: store.append(o))]
        for blk in model.blocks: hs.append(blk.register_forward_hook(lambda m, i, o: store.append(o)))
        S, Ys = [], []; ar = torch.arange(rows.shape[1], device=dev)
        try:
            for i in range(0, rows.shape[0], bs):
                x = torch.from_numpy(rows[i:i + bs].astype(np.int64)).to(dev); nr = torch.from_numpy(nreal[i:i + bs]).to(dev); store.clear(); model(x)
                H = torch.stack(store, 0).float()                                   # [L+1, B, T, d]
                m = (ar[None, :] >= 1) & (ar[None, :] < nr[:, None])               # rendering tokens only, BOS and padding excluded
                S.append(H[:, m].cpu().numpy()); Ys.append(labs[i:i + bs][m.cpu().numpy()])
        finally:
            for h in hs: h.remove()
        return np.concatenate(S, 1), np.concatenate(Ys, 0)

    def load_model(root, g, cond, d, seed, dev, n_head=4, n_layer=12, untrained=False):
        if untrained: torch.manual_seed(seed + 1000)
        model = X.TinyRotaryGPT(X.VOCAB, d, n_layer, n_head, X.SEQ_LEN + 1).to(dev)
        if not untrained: model.load_state_dict(torch.load(X.joint_paths(root, g, cond, d, seed)[1], map_location=dev)["model_state"])
        return model

    def run_one(root, g, d, seed, device, untrained=False):
        dev = torch.device(device); path = os.path.join(root, "depth", f"{g}{'_untrained' if untrained else ''}_d{d}_s{seed}.json")
        if os.path.exists(path): print("done already:", path); return
        model = load_model(root, g, "shared", d, seed, dev, untrained=untrained); grams = X.make_grammars("shared"); specs = {"A": {}, "A2": {}, "G": X.TRANSFORMS[g]}
        S, Y = {}, {}
        for lg in ("A", "A2", "G"):
            S[lg], Y[lg] = {}, {}
            for sp, n, sd in (("train", N_TRAIN, TRAIN_SEED), ("test", N_TEST, TEST_SEED)):
                rows, nreal, labs = make_rows_lab(grams[lg], specs[lg], X.BOS[lg], n, sd); S[lg][sp], Y[lg][sp] = layer_states(model, rows, nreal, labs, dev)
        res = dict(g=g, d=d, seed=seed, untrained=untrained, levels={})
        for k in PROBE_LEVELS:
            Yk = {lg: {sp: Y[lg][sp][:, k - 1].astype(int) for sp in Y[lg]} for lg in Y}
            for centered in (True, False):
                r = probe_transfer(S, Yk, centered=centered); r["onset"] = onset_layer(r["T"]); res["levels"][f"{k}_{'centered' if centered else 'strict'}"] = r
        os.makedirs(os.path.dirname(path), exist_ok=True); json.dump(res, open(path, "w")); print("wrote", path, "level-4 centered T by layer:", np.round(res["levels"]["4_centered"]["T"], 2).tolist(), flush=True)

# ----------------------------------------------------------------------------- summary and verdict (numpy only)
def summarize(root, d=32, seeds=(0,), level=PRIMARY_LEVEL, mode="centered"):
    import glob
    key = f"{level}_{mode}"; per = {}
    for g in X.G_LIST:
        runs = [json.load(open(p)) for s in seeds for p in glob.glob(os.path.join(root, "depth", f"{g}_d{d}_s{s}.json"))]
        if runs: per[g] = dict(onset=[r["levels"][key]["onset"] for r in runs], T=np.nanmean([r["levels"][key]["T"] for r in runs], 0), acc_AA2=np.mean([r["levels"][key]["acc_AA2"] for r in runs], 0),
                      dec=decodable_layer(np.mean([r["levels"][key]["acc_GG"] for r in runs], 0), runs[0]["levels"][key]["chance"][0]))
    _, T = load_all(root); A = cost_a(T, d, list(seeds)); gs = [g for g in per if g in A]
    print(f"=== CHECK 3: depth of sharing (level {level}, {mode} probes, d={d}, seeds={list(seeds)}) ===")
    if len(gs) < 4: print("VERDICT: NO VERDICT YET - fewer than 4 renderings have both a(g) and a depth result."); return
    print(f"\n{'g':13s} {'a':>7s} {'onset':>6s} {'decod.':>6s} {'mean T':>7s}  T by layer 0..{len(per[gs[0]]['T']) - 1}   (decod. = first layer where g's own probe works)")
    for g in sorted(gs, key=lambda g: A[g]):
        on = float(np.nanmedian(per[g]["onset"])) if np.isfinite(per[g]["onset"]).any() else float("nan")
        print(f"{g:13s} {A[g]:7.4f} {on:6.1f} {per[g]['dec']:6.1f} {np.nanmean(per[g]['T']):7.2f}  " + " ".join("  . " if not np.isfinite(t) else f"{t:4.1f}" for t in per[g]["T"]))
        per[g]["on"] = on
    ok = [g for g in gs if np.isfinite(per[g]["on"])]; sanity = np.mean([per[g]["acc_AA2"][-1] for g in gs])
    print(f"\nsanity: A-probe on A' (same order, new tag) at the last layer = {sanity:.2f} (should be near the A-on-A accuracy)")
    if len(ok) < C3_MIN_DEFINED: print(f"VERDICT: UNTESTABLE - only {len(ok)} renderings reach T >= {ONSET_FRAC}; fewer than {C3_MIN_DEFINED}. Look at mean T instead, and at the shared-model gate."); return
    a = [A[g] for g in ok]; on = [per[g]["on"] for g in ok]; tau = [string_baselines(g)["kendall_tau"] for g in ok]
    rho, prt = spearman(a, on), partial_rank_corr(a, on, [tau]); rmt = spearman([A[g] for g in gs], [-np.nanmean(per[g]["T"]) for g in gs])
    lag = [per[g]["on"] - per[g]["dec"] for g in ok]; rlag = spearman(a, lag) if len(ok) >= 4 else float("nan")
    print(f"Spearman(a, onset) = {rho:+.2f} (n={len(ok)});  partial given tau = {prt:+.2f};  secondary: Spearman(a, -mean T) = {rmt:+.2f} (n={len(gs)}),  Spearman(a, onset - decodable layer) = {rlag:+.2f}")
    v = "PASS" if (rho >= C3_PASS_RHO and prt >= C3_PASS_PARTIAL) else "FAIL" if rho <= C3_FAIL_RHO else "INCONCLUSIVE"
    print(f"VERDICT: {v}" + ("  <- sharing starts later for costlier renderings, beyond string distance" if v == "PASS" else "  <- depth of sharing does not track a(g)" if v == "FAIL" else ""))
    print("Reminder: this is probe transfer (shared readout), not causal patching. A PASS motivates the patching version; it does not replace it.")

In [28]:
j = json.load(open(f"{ROOT_LONG}/translator/relabel_new_d16_s0.json"))
c = j["curve"]; idx = [int(i*(len(c)-1)/9) for i in range(10)]
print("heldout_acc:", j["heldout_acc"], "| final_train:", j["final_train"])
print("curve:", [round(c[i][-1] if isinstance(c[i], (list,tuple)) else c[i], 4) for i in idx])
# compare against relabel_perm
j2 = json.load(open(f"{ROOT_LONG}/translator/relabel_perm_d16_s0.json"))
c2 = j2["curve"]; idx2 = [int(i*(len(c2)-1)/9) for i in range(10)]
print("relabel_perm curve:", [round(c2[i][-1] if isinstance(c2[i], (list,tuple)) else c2[i], 4) for i in idx2])

heldout_acc: 0.9852419154449611 | final_train: 0.13913764728419484
curve: [1.1449, 0.0909, 0.372, 0.1419, 0.1164, 0.1399, 0.1765, 0.1765, 0.1055, 0.1308]
relabel_perm curve: [1.1534, 0.0529, 0.025, 0.005, 0.0036, 0.0574, 0.0196, 0.081, 0.2919, 0.0124]


In [29]:
for g in ("relabel_new", "relabel_perm", "marker"):
    j = json.load(open(f"{ROOT_LONG}/translator/{g}_d16_s0.json"))
    print(g, "heldout_ce:", j["heldout_ce"], "heldout_acc:", j["heldout_acc"])

relabel_new heldout_ce: 0.04497100349635617 heldout_acc: 0.9852419154449611
relabel_perm heldout_ce: 0.0016095830750902944 heldout_acc: 0.9995740300308829
marker heldout_ce: 0.0001504052294053436 heldout_acc: 0.9999467537538603


In [30]:
# needs a new launch — one translator job, different seed, same d/steps
lines = [f"{sys.executable} exp8_lib.py translator --g relabel_new --seed 1 --d 16 --steps 72000 --root {ROOT_LONG} --device {DEVICE} > {ROOT_LONG}/logs/tr_relabel_new_d16_s1.txt 2>&1"]
launch(lines, "tr_relabel_new_seed1", ROOT_LONG)

1 jobs started, pid 3571099 | check progress with the status cells, not by guessing


In [70]:
import os, json
p = "./exp8_long/translator/relabel_new_d16_s1.json"
if os.path.exists(p):
    j = json.load(open(p))
    print("DONE. seed 1 heldout_ce:", j["heldout_ce"], "heldout_acc:", j["heldout_acc"])
else:
    print("not finished yet. last log lines:")
    print(os.popen("tail -n 10 ./exp8_long/logs/tr_relabel_new_d16_s1.txt 2>&1").read())
    print("still running:", os.popen("ps aux | grep exp8_lib | grep -v grep | wc -l").read().strip(), "exp8_lib processes")

DONE. seed 1 heldout_ce: 0.061649024467499404 heldout_acc: 0.978976607149196


In [31]:
print("VOCAB:", X.VOCAB)

VOCAB: 13


In [32]:
print("VOCAB:", X.VOCAB)
print("BOS:", X.BOS)
import inspect
print(inspect.getsource(X.build_grammar))

VOCAB: 13
BOS: {'A': 9, 'A2': 10, 'G': 11}
def build_grammar(seed):
    rng = random.Random(seed)
    symbols_by_level = {1: [(1, 0)]}
    for l in range(2, LEVELS + 1): symbols_by_level[l] = [(l, i) for i in range(NT_SIZE)]
    terminals = list(range(TERM_SIZE)); rules = defaultdict(list)
    for l in range(1, LEVELS + 1):
        child_level = l + 1
        child_symbols = terminals if child_level == LEVELS + 1 else symbols_by_level[child_level]
        for sym in symbols_by_level[l]:
            degree = rng.randint(MIN_DEGREE, MAX_DEGREE); seen = set(); tries = 0
            while len(rules[sym]) < degree and tries < 200:
                tries += 1; rule_len = rng.choice(RULE_LEN_CHOICES)
                rhs = tuple(rng.choice(child_symbols) for _ in range(rule_len))
                if rhs in seen: continue
                seen.add(rhs); rules[sym].append(rhs)
    return rules, symbols_by_level



In [37]:
print(os.popen(f"tail -n 3 {ROOT_LONG}/logs/tr_relabel_new_d16_s1.txt").read())

In [35]:
lines = [f"{sys.executable} exp8_lib.py translator --g relabel_new --seed 1 --d 16 --steps 72000 --root {ROOT_LONG} --device {DEVICE} > {ROOT_LONG}/logs/tr_relabel_new_d16_s1.txt 2>&1"]
launch(lines, "tr_relabel_new_seed1", ROOT_LONG)

1 jobs started, pid 3573259 | check progress with the status cells, not by guessing


In [33]:
import inspect
print(inspect.getsource(X.make_grammars))

def make_grammars(cond, names=LANGS3):
    shared, sbl = build_grammar(GRAMMAR_SEED)
    if cond == "shared": return {n: shared for n in names}
    assert cond == "disjoint"
    return {n: permute_grammar(shared, sbl, 200 + i) for i, n in enumerate(names)}



In [46]:
# does relabel_new's target vocabulary actually see fewer training examples?
import collections
rules, sbl = X.build_grammar(X.GRAMMAR_SEED)
rng = random.Random(0)
term_counts = collections.Counter()
for _ in range(5000):
    t = X.sample_tree(rules, rng)
    toks = X.render(t, X.TRANSFORMS["relabel_new"])
    term_counts.update(tok for tok in toks if tok in (4,5,6,7))
    toks_id = X.render(t, {})
    term_counts.update(("orig_"+str(tok)) if tok in (0,1,2,3) else None for tok in toks_id if tok in (0,1,2,3))
print(term_counts)

Counter({4: 565689, 'orig_0': 565689, 6: 285030, 'orig_2': 285030, 5: 197254, 'orig_1': 197254, 7: 135980, 'orig_3': 135980})


In [26]:
import exp8_lib as X
import pprint
pprint.pprint(X.TRANSFORMS)

{'identity': {},
 'marker': {'marker': 8},
 'perm_sym': {'perm_sym': {((1, 0), 2): [1, 0],
                           ((1, 0), 3): [0, 2, 1],
                           ((2, 0), 2): [1, 0],
                           ((2, 0), 3): [0, 2, 1],
                           ((2, 1), 2): [1, 0],
                           ((2, 1), 3): [1, 2, 0],
                           ((2, 2), 2): [1, 0],
                           ((2, 2), 3): [1, 0, 2],
                           ((2, 3), 2): [1, 0],
                           ((2, 3), 3): [1, 0, 2],
                           ((3, 0), 2): [1, 0],
                           ((3, 0), 3): [2, 0, 1],
                           ((3, 1), 2): [1, 0],
                           ((3, 1), 3): [1, 0, 2],
                           ((3, 2), 2): [1, 0],
                           ((3, 2), 3): [1, 2, 0],
                           ((3, 3), 2): [1, 0],
                           ((3, 3), 3): [2, 1, 0],
                           ((4, 0), 2): [1, 0],
                  

In [47]:
print("relabel_new min:", min(0.0909, 0.372, 0.1419, 0.1164, 0.1399, 0.1765, 0.1765, 0.1055, 0.1308))
print("relabel_perm min:", min(0.0529, 0.025, 0.005, 0.0036, 0.0574, 0.0196, 0.081, 0.2919, 0.0124))

relabel_new min: 0.0909
relabel_perm min: 0.0036


In [27]:
for g in X.G_LIST:
    b = string_baselines(g)
    print(f"{g:13s} tau={b['kendall_tau']:.3f}  disp={b['displacement']:.3f}")

relabel_new   tau=0.000  disp=0.000
relabel_perm  tau=0.000  disp=0.000
marker        tau=0.000  disp=0.000
swap5         tau=0.009  disp=0.009
rev6          tau=0.007  disp=0.005
rev5          tau=0.018  disp=0.014
rev4          tau=0.039  disp=0.033
rev45         tau=0.057  disp=0.034
rev_all       tau=1.000  disp=0.500
perm_sym      tau=0.595  disp=0.348


## STEP 1d. New helpers: launching, completeness checks, validity, gate table, probes

In [5]:
import subprocess, shutil, math
import numpy as np

def ensure_dirs(root):
    for sub in ("joint", "translator", "logs", "probe"): os.makedirs(os.path.join(root, sub), exist_ok=True)

def n_running():
    return int(subprocess.run("ps -eo args | grep exp8_lib.py | grep -v grep | wc -l", shell=True, capture_output=True, text=True).stdout.strip() or 0)

def launch(lines, name, root, parallel=8):
    if not lines: print("nothing to launch (all expected files already exist)"); return
    ensure_dirs(root); sh = f"{root}/{name}_jobs.sh"; open(sh, "w").write("\n".join(lines) + "\n")
    p = subprocess.Popen(f"cat {sh} | xargs -P {parallel} -I{{}} bash -c '{{}}'", shell=True, start_new_session=True, stdout=open(f"{root}/{name}.log", "a"), stderr=subprocess.STDOUT)
    print(len(lines), "jobs started, pid", p.pid, "| check progress with the status cells, not by guessing")

def translator_lines(root, d, seed, steps, gs, device):
    return [f"{sys.executable} exp8_lib.py translator --g {g} --seed {seed} --d {d} --steps {steps} --root {root} --device {device} > {root}/logs/tr_{g}_d{d}_s{seed}.txt 2>&1"
            for g in gs if not os.path.exists(X.tr_paths(root, g, d, seed))]

def joint_lines(root, d, seed, gs, conds, device):
    return [f"{sys.executable} exp8_lib.py joint --g {g} --cond {c} --seed {seed} --d {d} --steps 90000 --big_n 500000 --root {root} --device {device} > {root}/logs/jt_{g}_{c}_d{d}_s{seed}.txt 2>&1"
            for g in gs for c in conds if not os.path.exists(X.joint_paths(root, g, c, d, seed)[0])]

def joint_missing(root, d, seeds, gs, conds=("shared", "disjoint")):
    """counts JSON result files only (the .pt checkpoints are a separate file and must not be counted)"""
    return [(g, c, s) for g in gs for c in conds for s in seeds if not os.path.exists(X.joint_paths(root, g, c, d, s)[0])]

def status(root, d, seeds, gs):
    miss = joint_missing(root, d, seeds, gs); total = len(gs) * 2 * len(seeds)
    print(f"d={d}: {total - len(miss)} of {total} joint JSONs present | exp8_lib processes running: {n_running()}")
    if miss: print("  missing:", miss)
    return not miss

# ---------------------------------------------------------------- V1: is the cost measure valid at this d?
def translator_validity(root, d, seed=0, acc_min=0.99, verbose=True):
    need = ["identity"] + X.G_LIST; miss = [g for g in need if not os.path.exists(X.tr_paths(root, g, d, seed))]
    if miss: print(f"d={d}: translators missing {miss} | running: {n_running()}. Wait, do not read yet."); return None
    T = {g: json.load(open(X.tr_paths(root, g, d, seed))) for g in need}; a = {g: T[g]["auc"] - T["identity"]["auc"] for g in X.G_LIST}
    if verbose:
        print(f"d={d} translators, steps={T['identity']['steps']}\n{'g':13s} {'auc':>8s} {'heldout_acc':>12s} {'a(g)=auc-identity':>18s}")
        for g in need: print(f"{g:13s} {T[g]['auc']:8.4f} {T[g].get('heldout_acc', float('nan')):12.4f} {a.get(g, 0.0):18.4f}")
    id_ok = T["identity"].get("heldout_acc", 0) >= acc_min; neg = [g for g, v in a.items() if v < 0]; weak = [g for g in X.G_LIST if T[g].get("heldout_acc", 0) < 0.95]
    print(f"\nV1 (i) identity heldout_acc >= {acc_min}: {id_ok} | (ii) renderings with NEGATIVE a(g): {neg or 'none'} | not-a-criterion: renderings below 0.95 acc (cost may be undertrained): {weak or 'none'}")
    ok = id_ok and not neg; print("V1 =", "PASS: cost measure is usable at this d" if ok else "FAIL: do not run joint grids at this d"); return ok

def compare_long(root, root_long, d=16, seed=0):
    need = ["identity"] + X.G_LIST
    if any(not os.path.exists(X.tr_paths(r, g, d, seed)) for r in (root, root_long) for g in need): print("long-run translators not all finished yet | running:", n_running()); return
    S, L = {g: json.load(open(X.tr_paths(root, g, d, seed))) for g in need}, {g: json.load(open(X.tr_paths(root_long, g, d, seed))) for g in need}
    aS = {g: S[g]["auc"] - S["identity"]["auc"] for g in X.G_LIST}; aL = {g: L[g]["auc"] - L["identity"]["auc"] for g in X.G_LIST}
    print(f"{'g':13s} {'acc 24k':>8s} {'acc long':>9s} {'a 24k':>8s} {'a long':>8s}   (long steps = {L['identity']['steps']})")
    for g in sorted(X.G_LIST, key=lambda g: aL[g]): print(f"{g:13s} {S[g]['heldout_acc']:8.3f} {L[g]['heldout_acc']:9.3f} {aS[g]:8.4f} {aL[g]:8.4f}")
    print("Spearman(a_24k, a_long) =", round(spearman([aS[g] for g in X.G_LIST], [aL[g] for g in X.G_LIST]), 3), "| identity acc long:", round(L["identity"]["heldout_acc"], 4))
    print("If accuracies rise toward 1 and the ranking changes, the old costs were partly undertraining; use the long-run a(g).")

# ---------------------------------------------------------------- gate and per-rendering table, only on a complete grid
def read_gate(root, d, seeds, gs):
    if not status(root, d, seeds, gs): print("NOT COMPLETE: no gate read. Re-run this cell when every file is present."); return None
    J, _ = load_all(root); G = {k: v for k, v in gains(J, d, list(seeds)).items() if k[0] in gs}
    print(f"\n{'g':13s} {'seed':>4s} {'gain_A2':>8s} {'se_A2':>7s} {'gain_G':>8s} {'se_G':>7s} {'z_G':>6s} {'s':>7s}   excess shared / disjoint (A2)")
    for (g, s), v in sorted(G.items()):
        print(f"{g:13s} {s:>4d} {v['gain']['A2']:+8.4f} {v['se']['A2']:7.4f} {v['gain']['G']:+8.4f} {v['se']['G']:7.4f} {v['gain']['G'] / max(v['se']['G'], 1e-9):+6.1f} {v['s']:+7.3f}   {v['excess_shared']['A2']:+.4f} / {v['excess_disjoint']['A2']:+.4f}")
    ok, msg = gate_status(G); print(f"\nGATE d={d}: {msg} -> {'PASS' if ok else 'FAIL (untestable, not negative)'}"); return ok

# ---------------------------------------------------------------- amendment 2 check (read only)
def amendment_status(root):
    p = f"{root}/prereg_three_checks.txt"
    if not os.path.exists(p): print("no prereg_three_checks.txt at", p); return
    t = open(p).read(); print("AMENDMENT 1 recorded:", "AMENDMENT 1" in t, "| AMENDMENT 2 recorded:", "AMENDMENT 2" in t)

# ---------------------------------------------------------------- prereg v2 (new file, refuses to overwrite)
PREREG_V2 = """H-adapter pre-registration v2 - written {when}
Written AFTER: exact-floor fix; d=32 and d=16 gates read (no headroom, UNTESTABLE); d=8 gate PASS; Check 1 at d=8 FAIL on 3 seeds (diff -1.6 to -2.3);
d=8 translators found degenerate (identity heldout_acc 0.65, every a(g) negative). Nothing below was chosen after seeing any d=12 joint result or any probe-transfer result.

V1 (cost valid at a given d): all 11 translators exist; identity heldout_acc >= 0.99; no rendering has negative a(g).
V2 (headroom at a given d): gate_status PASS on the exact floor, read only when every expected joint JSON exists.
V1 must PASS at d before any joint run at that d. If V1 fails at d=12, try d=10 or d=14 once; if none passes, report that no capacity tested satisfies V1 and V2.

CHECK 1 and CHECK 2 at d=12: thresholds and definitions exactly as in prereg_three_checks.txt (frozen). They count only if V1 and V2 both PASS at d=12.

PROBE (new). d=16, seed 0, shared joint checkpoints, level-4 ancestor, centered ridge probes, cost a(g) from the 72k-step translators (24k-step values reported as secondary).
  dT(g) = mean over layers of T(g) on the trained shared model minus the same quantity on an untrained model of identical architecture.
  Prediction: cheap renderings transfer more, so Spearman(a, dT) is NEGATIVE.
  PASS <= -0.5 | FAIL >= -0.2 | INCONCLUSIVE otherwise | UNTESTABLE if fewer than 6 renderings have a defined dT.
  Reported, not part of the verdict: partial given tau, onset layer, the disjoint-model probe (its label space may not align with A's, see STEP 7c).
Strongest claim allowed if everything passes: consistent with the hypothesis in a synthetic setting. Failures get reported.
"""
def write_prereg_v2(root):
    p = os.path.join(root, "prereg_v2.txt")
    if os.path.exists(p): print("already frozen:", p); print(open(p).read()); return
    os.makedirs(root, exist_ok=True); open(p, "w").write(PREREG_V2.format(when=time.strftime("%Y-%m-%d %H:%M"))); print("wrote", p)

# ---------------------------------------------------------------- probes with cond + untrained baseline
def _n_layer_of(root, g, d, seed):
    sd = torch.load(X.joint_paths(root, g, "shared", d, seed)[1], map_location="cpu")["model_state"]
    return 1 + max(int(k.split(".")[1]) for k in sd if k.startswith("blocks."))

def run_probe(root, g, cond, d, seed, device, untrained=False):
    dev = torch.device(device); tag = "untrained" if untrained else cond; path = os.path.join(root, "probe", f"{g}_{tag}_d{d}_s{seed}.json")
    if os.path.exists(path): return
    if not os.path.exists(X.joint_paths(root, g, "shared", d, seed)[1]): print("no shared checkpoint yet:", g); return
    if not untrained and not os.path.exists(X.joint_paths(root, g, cond, d, seed)[1]): print("no checkpoint:", g, cond); return
    nl = _n_layer_of(root, g, d, seed); model = load_model(root, g, cond, d, seed, dev, n_layer=nl, untrained=untrained)
    grams = X.make_grammars("shared" if untrained else cond); specs = {"A": {}, "A2": {}, "G": X.TRANSFORMS[g]}; S, Y = {}, {}
    for lg in ("A", "A2", "G"):
        S[lg], Y[lg] = {}, {}
        for sp, n, sd in (("train", N_TRAIN, TRAIN_SEED), ("test", N_TEST, TEST_SEED)):
            rows, nreal, labs = make_rows_lab(grams[lg], specs[lg], X.BOS[lg], n, sd); S[lg][sp], Y[lg][sp] = layer_states(model, rows, nreal, labs, dev)
    res = dict(g=g, d=d, seed=seed, cond=tag, n_layer=nl, levels={})
    for k in PROBE_LEVELS:
        Yk = {lg: {sp: Y[lg][sp][:, k - 1].astype(int) for sp in Y[lg]} for lg in Y}
        r = probe_transfer(S, Yk, centered=True); r["onset"] = onset_layer(r["T"]); res["levels"][f"{k}_centered"] = r
    os.makedirs(os.path.dirname(path), exist_ok=True); json.dump(res, open(path, "w"))
    print("wrote", os.path.basename(path), "T(level 4) by layer:", np.round(res["levels"]["4_centered"]["T"], 2).tolist(), flush=True)

def _meanT(root, g, tag, d, seed, level=4):
    p = os.path.join(root, "probe", f"{g}_{tag}_d{d}_s{seed}.json")
    if not os.path.exists(p): return None
    r = json.load(open(p))["levels"][f"{level}_centered"]; T = np.asarray(r["T"], float)
    return dict(meanT=float(np.nanmean(T)) if np.isfinite(T).any() else float("nan"), n_def=int(np.isfinite(T).sum()), onset=r["onset"], gg=float(r["acc_GG"][-1]), chance=float(r["chance"][0]))

def probe_summary(root, d=16, seed=0, cost_root=None, level=4):
    cost_root = cost_root or root; _, TRc = load_all(cost_root); A = cost_a(TRc, d, [seed]); rows = []
    for g in X.G_LIST:
        sh, un, dj = _meanT(root, g, "shared", d, seed, level), _meanT(root, g, "untrained", d, seed, level), _meanT(root, g, "disjoint", d, seed, level)
        if sh and un and g in A and np.isfinite(sh["meanT"]) and np.isfinite(un["meanT"]): rows.append((g, A[g], sh, un, dj))
    print(f"=== PROBE: transfer of an A-trained level-{level} probe to rendering g (d={d}, seed {seed}; costs from {cost_root}) ===")
    print(f"{'g':13s} {'a(g)':>7s} {'meanT shared':>13s} {'meanT untrained':>16s} {'dT':>7s} {'#layers T defined':>18s} {'g->g acc (last)':>16s} {'meanT disjoint*':>16s}")
    for g, a, sh, un, dj in sorted(rows, key=lambda r: r[1]):
        print(f"{g:13s} {a:7.4f} {sh['meanT']:13.3f} {un['meanT']:16.3f} {sh['meanT'] - un['meanT']:+7.3f} {sh['n_def']:>18d} {sh['gg']:16.3f} {dj['meanT'] if dj else float('nan'):16.3f}")
    print("* disjoint column is descriptive only: its label space may not align with A's (see STEP 7c).")
    if len(rows) < 6: print(f"\nVERDICT: UNTESTABLE / NO VERDICT YET - {len(rows)} renderings with a defined dT, need 6."); return
    a = [r[1] for r in rows]; dT = [r[2]["meanT"] - r[3]["meanT"] for r in rows]; tau = [string_baselines(r[0])["kendall_tau"] for r in rows]
    rho = spearman(a, dT); prt = partial_rank_corr(a, dT, [tau]); rho_tau = spearman(tau, dT)
    v = "PASS" if rho <= -0.5 else "FAIL" if rho >= -0.2 else "INCONCLUSIVE"
    print(f"\nSpearman(a, dT) = {rho:+.2f} (n={len(rows)}) | partial given tau = {prt:+.2f} | baseline Spearman(tau, dT) = {rho_tau:+.2f}")
    print(f"VERDICT (prereg v2): {v}" + ("  <- cheaper renderings transfer more" if v == "PASS" else "  <- cost does not track probe transfer" if v == "FAIL" else ""))
    print("n<=10, one seed, probe not patching. Even a PASS is suggestive, not proof.")

In [7]:
HEADS = {12: 3}     # d=12 needs 3 heads (head_dim 4); every other d keeps the default of 4

def translator_lines(root, d, seed, steps, gs, device):
    nh = f" --n_head {HEADS[d]}" if d in HEADS else ""
    return [f"{sys.executable} exp8_lib.py translator --g {g} --seed {seed} --d {d} --steps {steps} --root {root} --device {device}{nh} > {root}/logs/tr_{g}_d{d}_s{seed}.txt 2>&1"
            for g in gs if not os.path.exists(X.tr_paths(root, g, d, seed))]

def joint_lines(root, d, seed, gs, conds, device):
    nh = f" --n_head {HEADS[d]}" if d in HEADS else ""
    return [f"{sys.executable} exp8_lib.py joint --g {g} --cond {c} --seed {seed} --d {d} --steps 90000 --big_n 500000 --root {root} --device {device}{nh} > {root}/logs/jt_{g}_{c}_d{d}_s{seed}.txt 2>&1"
            for g in gs for c in conds if not os.path.exists(X.joint_paths(root, g, c, d, seed)[0])]

open(f"{ROOT}/prereg_v2.txt", "a").write(f"\nNOTE ({time.strftime('%Y-%m-%d %H:%M')}): d=12 with 4 heads fails exp8_lib's RoPE assert (head_dim 3 is odd). d=12 now runs with --n_head 3 (head_dim 4). Decided before any d=12 result. Criteria V1/V2 unchanged.\n")
print("d=12 jobs will use --n_head 3;", translator_lines(ROOT, 12, 0, 24000, ["identity"], DEVICE)[0][-120:])

IndexError: list index out of range

## STEP 2. Config and flags (all launches off by default)

In [6]:
ROOT      = "./exp8"            # existing results (d=8, 16, 32) live here; new d=12 files go here too
ROOT_LONG = "./exp8_long"       # NEW separate folder for 72k-step d=16 translators, so nothing in ROOT is overwritten
DEVICE    = "cuda:2"            # check nvidia-smi first
D_NEW, D_LONG_TR, LONG_STEPS, STD_STEPS = 12, 16, 72000, 24000
SEEDS = [0, 7, 13]
FIRST_PASS = ["marker", "rev4", "rev45", "rev_all"]     # enough for Check 1. Use X.G_LIST once the first pass looks sane (Check 2 needs >= 8)
D12_SET = FIRST_PASS

LAUNCH_D12_TRANSLATORS = True   # 4a
LAUNCH_D16_LONG        = True   # 4b
LAUNCH_D16_SHARED      = True   # 4c  finish the 4 missing d=16 shared joints (probes need shared only; the untrained model is the control)
LAUNCH_D12_JOINT       = True   # 6a  refuses unless V1 passes at d=12
LAUNCH_D12_SEEDS       = True   # 8a  seeds 7 and 13, only after the d=12 gate PASSES
RUN_PROBES             = True   # 7b  GPU
print("ROOT:", ROOT, "| renderings:", X.G_LIST)

ROOT: ./exp8 | renderings: ['relabel_new', 'relabel_perm', 'marker', 'swap5', 'rev6', 'rev5', 'rev4', 'rev45', 'rev_all', 'perm_sym']


## STEP 3. Freeze prereg v2 (once, BEFORE any d=12 joint or probe result is read) and check Amendment 2

In [6]:
write_prereg_v2(ROOT)
amendment_status(ROOT)      # if AMENDMENT 2 is False: set RECORD_AMENDMENT_2 in the old notebook, or append the text by hand. Not urgent tonight.

wrote ./exp8/prereg_v2.txt
AMENDMENT 1 recorded: True | AMENDMENT 2 recorded: False


## STEP 4. Phase A: launches (cheap, background, resumable; each skips files that exist)

In [8]:
# 4a. d=12 translators: identity + all 10 renderings, seed 0, 24k steps (same settings as the d=8/d=16 translators)
if not LAUNCH_D12_TRANSLATORS: print("skipped (flag False)")
else: launch(translator_lines(ROOT, D_NEW, 0, STD_STEPS, ["identity"] + X.G_LIST, DEVICE), "tr_d12", ROOT)

11 jobs started, pid 3547511 | check progress with the status cells, not by guessing


In [9]:
!tail -n 3 exp8/logs/tr_identity_d12_s0.txt

In [21]:
# 4b. d=16 translators trained 3x longer, ALL 11, into a separate root. Tests whether rev4/rev45/rev_all/perm_sym stopped at 83-86% because of undertraining.
# If a job fails immediately, read {ROOT_LONG}/logs/tr_identity_d16_s0.txt (exp8_lib may expect something in --root that ROOT has).
if not LAUNCH_D16_LONG: print("skipped (flag False)")
else:
    ensure_dirs(ROOT_LONG); launch(translator_lines(ROOT_LONG, D_LONG_TR, 0, LONG_STEPS, ["identity"] + X.G_LIST, DEVICE), "tr_d16_long", ROOT_LONG)

11 jobs started, pid 2978421 | check progress with the status cells, not by guessing


In [22]:
# 4c. finish the d=16 SHARED joint runs (the 4 renderings still missing). Shared only: the probes use an untrained baseline, not the disjoint model.
if not LAUNCH_D16_SHARED: print("skipped (flag False)")
else: launch(joint_lines(ROOT, 16, 0, X.G_LIST, ("shared",), DEVICE), "jt_d16_shared", ROOT)

nothing to launch (all expected files already exist)


## STEP 5. Phase B: is the cost measure valid? (read only when the files exist)

In [18]:
print(os.popen(f"ls {ROOT}/translator | grep d12; echo ---; tail -n 3 {ROOT}/logs/tr_identity_d12_s0.txt; echo ---; ps -eo pid,etime,args | grep 'exp8_lib.py translator' | grep -v grep | head -3").read())

---
  File "/home/anwoy/miniconda3/envs/ke/lib/python3.10/random.py", line 378, in choice
    return seq[self._randbelow(len(seq))]
KeyboardInterrupt
---



In [19]:
print(sorted(f for f in os.listdir(f"{ROOT}/translator") if "_d12_" in f))
print(os.popen("ps -eo pid,etime,args | grep 'exp8_lib.py translator' | grep -v grep | wc -l").read().strip(), "translator processes running")

[]
0 translator processes running


In [20]:
ensure_dirs(ROOT)
lines = [f"{sys.executable} exp8_lib.py translator --g {g} --seed 0 --d 12 --n_head 3 --steps 24000 --root {ROOT} --device {DEVICE} > {ROOT}/logs/tr_{g}_d12_s0.txt 2>&1"
         for g in ["identity"] + X.G_LIST if not os.path.exists(X.tr_paths(ROOT, g, 12, 0))]
print(lines[0])
launch(lines, "tr_d12", ROOT)

/home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py translator --g identity --seed 0 --d 12 --n_head 3 --steps 24000 --root ./exp8 --device cuda:2 > ./exp8/logs/tr_identity_d12_s0.txt 2>&1
11 jobs started, pid 3553170 | check progress with the status cells, not by guessing


In [63]:
print(os.popen(f"tail -n 3 {ROOT}/logs/tr_identity_d12_s0.txt; echo ---; ps -eo pid,etime,args | grep 'exp8_lib.py translator' | grep -v grep | wc -l").read())

step  15000 out-CE 0.1260
step  20000 out-CE 0.0210
{"g": "identity", "d": 12, "seed": 0, "steps": 24000, "auc": 0.16077051741339768, "final_train": 0.020624011236126533, "heldout_ce": 0.0022702742078248347, "heldout_acc": 0.9995562812821696, "win": 500}
---
9



In [65]:
print(len([f for f in os.listdir(f"{ROOT}/translator") if "_d12_" in f]), "of 11 d=12 translators finished |",
      int(os.popen("ps -eo args | grep 'python.* exp8_lib.py translator' | grep -v grep | grep -v 'bash -c' | wc -l").read()), "running")

11 of 11 d=12 translators finished | 0 running


In [66]:
# 5a. V1 at d=12. Prints a table and PASS/FAIL. Needs all 11 translators.
v1_d12 = translator_validity(ROOT, D_NEW)

d=12 translators, steps=24000
g                  auc  heldout_acc  a(g)=auc-identity
identity        0.1608       0.9996             0.0000
relabel_new     0.4047       0.9919             0.2439
relabel_perm    0.2272       0.9989             0.0665
marker          0.3050       0.8429             0.1443
swap5           0.4149       0.9035             0.2541
rev6            0.2444       0.9929             0.0836
rev5            0.4570       0.9094             0.2962
rev4            0.6248       0.7771             0.4640
rev45           0.6003       0.8063             0.4395
rev_all         0.3703       0.9905             0.2095
perm_sym        0.6890       0.7795             0.5282

V1 (i) identity heldout_acc >= 0.99: True | (ii) renderings with NEGATIVE a(g): none | not-a-criterion: renderings below 0.95 acc (cost may be undertrained): ['marker', 'swap5', 'rev5', 'rev4', 'rev45', 'perm_sym']
V1 = PASS: cost measure is usable at this d


In [79]:
v1_d12_long = translator_validity(f"{ROOT}_12long", D_NEW)

d=12: translators missing ['identity', 'relabel_new', 'relabel_perm', 'rev6', 'rev_all'] | running: 0. Wait, do not read yet.


In [67]:
def translator_validity(root, d, seed=0, acc_min=0.99, min_converged_frac=0.8, converged_acc=0.95, verbose=True):
    need = ["identity"] + X.G_LIST; miss = [g for g in need if not os.path.exists(X.tr_paths(root, g, d, seed))]
    if miss: print(f"d={d}: translators missing {miss} | running: {n_running()}. Wait, do not read yet."); return None
    T = {g: json.load(open(X.tr_paths(root, g, d, seed))) for g in need}; a = {g: T[g]["auc"] - T["identity"]["auc"] for g in X.G_LIST}
    if verbose:
        print(f"d={d} translators, steps={T['identity']['steps']}\n{'g':13s} {'auc':>8s} {'heldout_acc':>12s} {'a(g)':>8s}")
        for g in need: print(f"{g:13s} {T[g]['auc']:8.4f} {T[g].get('heldout_acc', float('nan')):12.4f} {a.get(g, 0.0):8.4f}")
    id_ok = T["identity"].get("heldout_acc", 0) >= acc_min
    neg = [g for g, v in a.items() if v < 0]
    unconverged = [g for g in X.G_LIST if T[g].get("heldout_acc", 0) < converged_acc]
    conv_frac = 1 - len(unconverged) / len(X.G_LIST)
    print(f"\nV1 (i) identity >= {acc_min}: {id_ok} | (ii) negative a(g): {neg or 'none'} | (iii) renderings >= {converged_acc} acc: {len(X.G_LIST)-len(unconverged)}/{len(X.G_LIST)} (need >= {min_converged_frac:.0%}) | unconverged: {unconverged or 'none'}")
    ok = id_ok and not neg and conv_frac >= min_converged_frac
    print("V1 =", "PASS: cost measure usable at this d" if ok else "FAIL: too many renderings undertrained, do not run joint grids at this d")
    return ok

print("retroactive check, d=12 24k-step data:")
translator_validity(ROOT, 12, verbose=False)

retroactive check, d=12 24k-step data:

V1 (i) identity >= 0.99: True | (ii) negative a(g): none | (iii) renderings >= 0.95 acc: 4/10 (need >= 80%) | unconverged: ['marker', 'swap5', 'rev5', 'rev4', 'rev45', 'perm_sym']
V1 = FAIL: too many renderings undertrained, do not run joint grids at this d


False

In [68]:
lines = translator_lines(ROOT, 12, 0, 72000, [g for g in ["identity"] + X.G_LIST if json.load(open(X.tr_paths(ROOT, g, 12, 0))).get("heldout_acc", 0) < 0.99], DEVICE)
launch(lines, "tr_d12_72k", ROOT)

nothing to launch (all expected files already exist)


In [69]:
unconverged = [g for g in ["identity"] + X.G_LIST if json.load(open(X.tr_paths(ROOT, g, 12, 0))).get("heldout_acc", 0) < 0.99]
print("relaunching at 72k steps:", unconverged)
lines = [f"{sys.executable} exp8_lib.py translator --g {g} --seed 0 --d 12 --steps 72000 --root {ROOT}_12long --device {DEVICE} --n_head 3 > {ROOT}_12long/logs/tr_{g}_d12_s0.txt 2>&1" for g in unconverged]
ensure_dirs(f"{ROOT}_12long")
launch(lines, "tr_d12_72k", f"{ROOT}_12long")

relaunching at 72k steps: ['marker', 'swap5', 'rev5', 'rev4', 'rev45', 'perm_sym']
6 jobs started, pid 3635418 | check progress with the status cells, not by guessing


In [77]:
print("exp8_lib processes running:", n_running())

exp8_lib processes running: 0


In [80]:
print(os.popen(f"tail -n 20 {ROOT}_12long/logs/tr_identity_d12_s0.txt").read())

tail: cannot open './exp8_12long/logs/tr_identity_d12_s0.txt' for reading: No such file or directory


In [81]:
unconverged = [g for g in ["identity"] + X.G_LIST if json.load(open(X.tr_paths(ROOT, g, 12, 0))).get("heldout_acc", 0) < 0.99]

In [83]:
need = ["identity"] + X.G_LIST
merged = {}
for g in need:
    p_long = X.tr_paths(f"{ROOT}_12long", g, 12, 0)
    p_std  = X.tr_paths(ROOT, g, 12, 0)
    merged[g] = json.load(open(p_long if os.path.exists(p_long) else p_std))
a = {g: merged[g]["auc"] - merged["identity"]["auc"] for g in X.G_LIST}
print(f"{'g':13s} {'steps':>6s} {'auc':>8s} {'heldout_acc':>12s} {'a(g)':>8s}")
for g in need: print(f"{g:13s} {merged[g]['steps']:>6d} {merged[g]['auc']:8.4f} {merged[g].get('heldout_acc', float('nan')):12.4f} {a.get(g, 0.0):8.4f}")
id_ok = merged["identity"]["heldout_acc"] >= 0.99
neg = [g for g in X.G_LIST if a[g] < 0]
print("\nid_ok:", id_ok, "| negative a(g):", neg or "none", "-> V1", "PASS" if id_ok and not neg else "FAIL")

g              steps      auc  heldout_acc     a(g)
identity       24000   0.1608       0.9996   0.0000
relabel_new    24000   0.4047       0.9919   0.2439
relabel_perm   24000   0.2272       0.9989   0.0665
marker         72000   0.2757       0.9999   0.1150
swap5          72000   0.2135       0.9720   0.0527
rev6           24000   0.2444       0.9929   0.0836
rev5           72000   0.2927       0.9536   0.1319
rev4           72000   0.5343       0.8106   0.3735
rev45          72000   0.4914       0.8389   0.3307
rev_all        24000   0.3703       0.9905   0.2095
perm_sym       72000   0.5572       0.8200   0.3964

id_ok: True | negative a(g): none -> V1 PASS


In [84]:
# does this rendering's target vocabulary include ids that rarely/never appear as the LOSS target (canonical side), unlike the input side?
import collections
for g in ("relabel_new", "marker", "relabel_perm", "rev4", "rev_all"):
    rows, nreal, labs = make_rows_lab(X.make_grammars("shared")["G"], X.TRANSFORMS[g], X.BOS["G"], 2000, 424242)
    # adjust indexing to your actual row layout: [BOS_G, g(z)..., SEP, canonical(z)...]
    print(g, "canonical-side (loss target) token ids seen:", sorted(set(rows[:, -5:].flatten().tolist()))[:15])

relabel_new canonical-side (loss target) token ids seen: [0]
marker canonical-side (loss target) token ids seen: [0]
relabel_perm canonical-side (loss target) token ids seen: [0]
rev4 canonical-side (loss target) token ids seen: [0]
rev_all canonical-side (loss target) token ids seen: [0]


In [85]:
rows, nreal, labs = make_rows_lab(X.make_grammars("shared")["G"], X.TRANSFORMS["relabel_new"], X.BOS["G"], 3, 424242)
print("rows shape:", rows.shape)
print(rows[0])
print(rows[1])
print(rows[2])
print("nreal[:3]:", nreal[:3])

rows shape: (3, 256)
[11  6  7  4  6  5  6  5  4  5  6  4  5  6  5  4  5  6  5  4  5  6  4  4
  6  5  6  5  4  4  4  4  7  4  4  7  6  4  4  5  6  4  4  4  5  6  6  7
  4  4  4  4  4  6  6  5  4  5  4  4  6  4  4  4  4  7  4  5  6  7  6  4
  6  5  4  4  6  7  4  6  6  4  4  7  4  4  6  5  4  5  6  6  5  4  7  6
  4  6  7  4  4  4  7  6  4  6  6  4  4  6  7  6  4  6  6  4  4  7  6  7
  4  4  4  4  4  6  7  4  6  5  4  4  4  5  6  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
[11  4  4  6  5  4  5  4  4  6  6  7  4  4  4  4  4  4  4  7  4  4  6  5
  5  4  4  4  4  4  6  7  6  4  6  5  6  7  4  4  4  4  4  5  4  5  4  4
  6  5  4  4  6  4  7  7  6  4  4  4  7  4  5  6  4  

In [86]:
import inspect
print(inspect.getsource(train_translator))

NameError: name 'train_translator' is not defined

In [88]:
j_mk = json.load(open(f"{ROOT}_12long/translator/marker_d12_s0.json"))
j_id = json.load(open(X.tr_paths(ROOT, "identity", 12, 0)))
print("marker heldout_acc:", j_mk["heldout_acc"], "| identity:", j_id["heldout_acc"])

marker heldout_acc: 0.9998846331333641 | identity: 0.9995562812821696


In [87]:
print(os.popen("sed -n '1,260p' /home/anwoy/exp8_lib.py").read())

"""Exp 8 library: adapter-cost hypothesis (H-adapter).
Claim: sharing between a rendering g(z) and the canonical rendering falls with a(g), the cost of converting g(z) into canonical(z)
using only the model's own primitives (causal RoPE attention + MLP).
  a(g): train the SAME architecture as a translator   [BOS g(z) SEP canonical(z)], loss on the canonical part only, fixed step budget.
  s(g): joint model on {A (canonical, 70%), A' (identical rendering, new tag, 15%), g (15%)}; compare against a DISJOINT-grammar control
        with the same data volume.  gain_L = CE_L(disjoint) - CE_L(shared)  (positive = sharing the tree helps language L);
        s(g) = gain_g / gain_A'   (A' is the ceiling: same order, same tree).
CLI:  python exp8_lib.py joint --g rev45 --cond shared --seed 0 --d 32 --steps 90000 --root ./exp8 --device cuda:0
      python exp8_lib.py translator --g rev45 --seed 0 --d 32 --steps 24000 --root ./exp8 --device cuda:0
Everything above the TORCH marker is torch-free (a

In [82]:
j = json.load(open(X.tr_paths(ROOT, "identity", 12, 0)))
print("identity at d=12, 24k steps: heldout_acc =", j.get("heldout_acc"))

identity at d=12, 24k steps: heldout_acc = 0.9995562812821696


In [48]:
# 5b. do the d=16 costs survive 3x longer training?
compare_long(ROOT, ROOT_LONG, d=16)
v1_d16_long = translator_validity(ROOT_LONG, 16)

g              acc 24k  acc long    a 24k   a long   (long steps = 72000)
rev6             0.998     0.998   0.0424   0.0142
marker           1.000     1.000   0.0545   0.0179
rev_all          0.835     0.998   0.1145   0.0444
swap5            0.978     0.989   0.1194   0.0643
relabel_perm     0.993     1.000   0.1175   0.0680
rev5             0.971     0.982   0.1730   0.0987
relabel_new      0.995     0.985   0.1183   0.1299
rev45            0.837     0.918   0.4370   0.3326
perm_sym         0.856     0.892   0.4318   0.3602
rev4             0.833     0.864   0.4132   0.3630
Spearman(a_24k, a_long) = 0.891 | identity acc long: 0.9998
If accuracies rise toward 1 and the ranking changes, the old costs were partly undertraining; use the long-run a(g).
d=16 translators, steps=72000
g                  auc  heldout_acc  a(g)=auc-identity
identity        0.0320       0.9998             0.0000
relabel_new     0.1619       0.9852             0.1299
relabel_perm    0.1000       0.9996         

## STEP 6. Phase C: d=12 joint first pass (the real test of claim 1)

In [91]:
# 6a. launches only if V1 passed at d=12 (checks the merged root: ./exp8 for already-converged renderings, ./exp8_12long for the four relaunched at 72k)
if not LAUNCH_D12_JOINT: print("skipped (flag False)")
else:
    need = ["identity"] + X.G_LIST
    paths = {g: (X.tr_paths(f"{ROOT}_12long", g, D_NEW, 0) if os.path.exists(X.tr_paths(f"{ROOT}_12long", g, D_NEW, 0)) else X.tr_paths(ROOT, g, D_NEW, 0)) for g in need}
    missing = [g for g in need if not os.path.exists(paths[g])]
    if missing: print("still missing:", missing, "- not launching")
    else:
        merged = {g: json.load(open(paths[g])) for g in need}
        a = {g: merged[g]["auc"] - merged["identity"]["auc"] for g in X.G_LIST}
        ok = merged["identity"]["heldout_acc"] >= 0.99 and not any(v < 0 for v in a.values())
        if not ok: print("V1 has not passed at d=12: not launching.")
        else: launch(joint_lines(ROOT, D_NEW, 0, D12_SET, ("shared", "disjoint"), DEVICE), "jt_d12_s0", ROOT)

8 jobs started, pid 3719561 | check progress with the status cells, not by guessing


In [ ]:
os.system("pkill -STOP -f exp8_lib.py")
print("stopped (not killed) — processes are frozen, not running, won't show CPU/GPU usage")

In [93]:
print("exp8_lib processes running:", n_running())
print(status(ROOT, D_NEW, [0], D12_SET))   # prints X of 8 JSONs present (4 renderings × shared/disjoint) and lists what's missing

exp8_lib processes running: 16
d=12: 0 of 8 joint JSONs present | exp8_lib processes running: 16
  missing: [('marker', 'shared', 0), ('marker', 'disjoint', 0), ('rev4', 'shared', 0), ('rev4', 'disjoint', 0), ('rev45', 'shared', 0), ('rev45', 'disjoint', 0), ('rev_all', 'shared', 0), ('rev_all', 'disjoint', 0)]
False


In [14]:
# 6b. completeness first, then the per-rendering table, then Check 1 (needs 3 seeds for a verdict; with seed 0 only it prints the diff and says NO VERDICT).
if read_gate(ROOT, D_NEW, [0], D12_SET):
    r1_d12 = check1(ROOT, d=D_NEW, seeds=[0])

d=12: 8 of 8 joint JSONs present | exp8_lib processes running: 0

g             seed  gain_A2   se_A2   gain_G    se_G    z_G       s   excess shared / disjoint (A2)
marker           0  -0.0000  0.0015  +0.0045  0.0009   +4.7    +nan   +0.0268 / +0.0268
rev4             0  -0.0022  0.0014  +0.0053  0.0015   +3.6    +nan   +0.0230 / +0.0209
rev45            0  +0.0044  0.0015  +0.0107  0.0014   +7.5    +nan   +0.0222 / +0.0266
rev_all          0  -0.0014  0.0015  -0.0029  0.0016   -1.8    +nan   +0.0289 / +0.0275

GATE d=12: mean gain_A' 0.0002 nats over 4 runs (SE of mean ~0.0007) -> FAIL (untestable, not negative)


In [15]:
import numpy as np

J, TR = load_all(ROOT)
Gr12 = {k: v for k, v in gains(J, D_NEW, [0]).items() if k[0] in D12_SET}

print(f"{'g':13s} {'gain_A2':>8s} {'se_A2':>7s} {'gain_G':>8s} {'se_G':>7s} {'z_G':>6s} {'s':>7s}   excess shared / disjoint (A2)")
for (g, s), v in sorted(Gr12.items()):
    print(f"{g:13s} {v['gain']['A2']:+8.4f} {v['se']['A2']:7.4f} {v['gain']['G']:+8.4f} {v['se']['G']:7.4f} {v['gain']['G']/max(v['se']['G'],1e-9):+6.1f} {v['s']:+7.3f}   {v['excess_shared']['A2']:+.4f} / {v['excess_disjoint']['A2']:+.4f}")

ok, msg = gate_status(Gr12)
print(f"\nGATE d=12: {msg} -> {'PASS' if ok else 'FAIL (untestable, not negative)'}")

if ok:
    r1_d12 = check1(ROOT, d=D_NEW, seeds=[0])
    print(r1_d12)

g              gain_A2   se_A2   gain_G    se_G    z_G       s   excess shared / disjoint (A2)
marker         -0.0000  0.0015  +0.0045  0.0009   +4.7    +nan   +0.0268 / +0.0268
rev4           -0.0022  0.0014  +0.0053  0.0015   +3.6    +nan   +0.0230 / +0.0209
rev45          +0.0044  0.0015  +0.0107  0.0014   +7.5    +nan   +0.0222 / +0.0266
rev_all        -0.0014  0.0015  -0.0029  0.0016   -1.8    +nan   +0.0289 / +0.0275

GATE d=12: mean gain_A' 0.0002 nats over 4 runs (SE of mean ~0.0007) -> FAIL (untestable, not negative)


In [5]:
print("still running:", n_running())

still running: 0


In [7]:
print(os.popen("ps aux | grep exp8_lib.py | grep -v grep").read())

anwoy    3719564  0.0  0.0   7344  3724 ?        T    04:56   0:00 bash -c /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py joint --g marker --cond shared --seed 0 --d 12 --steps 90000 --big_n 500000 --root ./exp8 --device cuda:2 --n_head 3 > ./exp8/logs/jt_marker_shared_d12_s0.txt 2>&1
anwoy    3719565  0.0  0.0   7344  3776 ?        T    04:56   0:00 bash -c /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py joint --g marker --cond disjoint --seed 0 --d 12 --steps 90000 --big_n 500000 --root ./exp8 --device cuda:2 --n_head 3 > ./exp8/logs/jt_marker_disjoint_d12_s0.txt 2>&1
anwoy    3719566  0.0  0.0   7344  3776 ?        T    04:56   0:00 bash -c /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py joint --g rev4 --cond shared --seed 0 --d 12 --steps 90000 --big_n 500000 --root ./exp8 --device cuda:2 --n_head 3 > ./exp8/logs/jt_rev4_shared_d12_s0.txt 2>&1
anwoy    3719567  6.6  0.1 44093752 1537560 ?    Tl   04:56  32:08 /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py 

In [6]:
print(os.popen("pkill -CONT -f exp8_lib.py").read())
print("still running:", n_running())


still running: 0


In [7]:
import time
print(os.popen(f"tail -n 2 {ROOT}/logs/jt_marker_shared_d12_s0.txt").read())
time.sleep(30)
print(os.popen(f"tail -n 2 {ROOT}/logs/jt_marker_shared_d12_s0.txt").read())

    super().__init__(); assert n_embd % n_head == 0 and (n_embd // n_head) % 2 == 0, "head_dim must be even for RoPE"
AssertionError: head_dim must be even for RoPE

    super().__init__(); assert n_embd % n_head == 0 and (n_embd // n_head) % 2 == 0, "head_dim must be even for RoPE"
AssertionError: head_dim must be even for RoPE



In [8]:
import sys, os, json, subprocess

ROOT, D_NEW, DEVICE = "./exp8", 12, "cuda:2"
D12_SET = ["marker", "rev4", "rev45", "rev_all"]
HEADS = {12: 3}

def n_running():
    return int(subprocess.run("ps -eo args | grep exp8_lib.py | grep -v grep | wc -l", shell=True, capture_output=True, text=True).stdout.strip() or 0)

def joint_path(root, g, c, d, seed):
    return os.path.join(root, "joint", f"{g}_{c}_d{d}_s{seed}.json")

def joint_lines(root, d, seed, gs, conds, device):
    nh = f" --n_head {HEADS[d]}" if d in HEADS else ""
    return [f"{sys.executable} exp8_lib.py joint --g {g} --cond {c} --seed {seed} --d {d} --steps 90000 --big_n 500000 --root {root} --device {device}{nh} > {root}/logs/jt_{g}_{c}_d{d}_s{seed}.txt 2>&1"
            for g in gs for c in conds if not os.path.exists(joint_path(root, g, c, d, seed))]

os.makedirs(f"{ROOT}/joint", exist_ok=True); os.makedirs(f"{ROOT}/logs", exist_ok=True)
print("currently running:", n_running())

lines = joint_lines(ROOT, D_NEW, 0, D12_SET, ("shared", "disjoint"), DEVICE)
print(f"{len(lines)} jobs to launch:")
for l in lines: print(" ", l)

currently running: 16
8 jobs to launch:
  /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py joint --g marker --cond shared --seed 0 --d 12 --steps 90000 --big_n 500000 --root ./exp8 --device cuda:2 --n_head 3 > ./exp8/logs/jt_marker_shared_d12_s0.txt 2>&1
  /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py joint --g marker --cond disjoint --seed 0 --d 12 --steps 90000 --big_n 500000 --root ./exp8 --device cuda:2 --n_head 3 > ./exp8/logs/jt_marker_disjoint_d12_s0.txt 2>&1
  /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py joint --g rev4 --cond shared --seed 0 --d 12 --steps 90000 --big_n 500000 --root ./exp8 --device cuda:2 --n_head 3 > ./exp8/logs/jt_rev4_shared_d12_s0.txt 2>&1
  /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py joint --g rev4 --cond disjoint --seed 0 --d 12 --steps 90000 --big_n 500000 --root ./exp8 --device cuda:2 --n_head 3 > ./exp8/logs/jt_rev4_disjoint_d12_s0.txt 2>&1
  /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py joint --g rev45 --cond s

In [9]:
sh = f"{ROOT}/jt_d12_retry2_jobs.sh"
open(sh, "w").write("\n".join(lines) + "\n")
p = subprocess.Popen(f"cat {sh} | xargs -P 8 -I{{}} bash -c '{{}}'", shell=True, start_new_session=True,
                      stdout=open(f"{ROOT}/jt_d12_retry2.log", "a"), stderr=subprocess.STDOUT)
print("started, pid", p.pid)

started, pid 496060


In [13]:
import os, json

def status_check(root, d, seeds, gs):
    have = []
    for g in gs:
        for c in ("shared", "disjoint"):
            for s in seeds:
                p = os.path.join(root, "joint", f"{g}_{c}_d{d}_s{s}.json")
                have.append((g, c, s, os.path.exists(p)))
    done = sum(1 for *_, ok in have if ok)
    total = len(have)
    print(f"{done} of {total} JSONs present")
    if done < total:
        print("missing:", [(g, c, s) for g, c, s, ok in have if not ok])
    print("processes running:", n_running())
    return done == total

status_check(ROOT, D_NEW, [0], D12_SET)

8 of 8 JSONs present
processes running: 0


True

In [11]:
print("still running:", n_running())
print(os.popen(f"tail -n 8 {ROOT}/logs/jt_marker_shared_d12_s0.txt").read())

still running: 16
step   45000 loss 0.7705
step   50000 loss 0.7701
step   55000 loss 0.7702
step   60000 loss 0.7697
step   65000 loss 0.7690
step   70000 loss 0.7686
step   75000 loss 0.7683
step   80000 loss 0.7685



In [11]:
print("still running:", n_running())  # confirm 0 before relaunching, avoid double-submitting
launch(joint_lines(ROOT, D_NEW, 0, D12_SET, ("shared", "disjoint"), DEVICE), "jt_d12_s0_retry", ROOT)

still running: 0
8 jobs started, pid 263415 | check progress with the status cells, not by guessing


## STEP 7. Phase D: probe transfer on d=16 shared checkpoints, with an untrained baseline

In [7]:
# 7a. status of the d=16 shared checkpoints (the probe needs one per rendering)
have = [g for g in X.G_LIST if os.path.exists(X.joint_paths(ROOT, g, "shared", 16, 0)[1])]
print(len(have), "of 10 shared d=16 checkpoints:", have, "| missing:", [g for g in X.G_LIST if g not in have], "| running:", n_running())

10 of 10 shared d=16 checkpoints: ['relabel_new', 'relabel_perm', 'marker', 'swap5', 'rev6', 'rev5', 'rev4', 'rev45', 'rev_all', 'perm_sym'] | missing: [] | running: 0


In [ ]:
# 7b. GPU: probe every rendering that has a checkpoint: trained shared model, untrained model of the same architecture, and the disjoint model if its checkpoint exists.
if not RUN_PROBES: print("skipped (RUN_PROBES False)")
else:
    for g in X.G_LIST:
        for cond, untr in (("shared", False), ("shared", True), ("disjoint", False)):
            run_probe(ROOT, g, cond, 16, 0, DEVICE, untrained=untr)

In [12]:
import json, numpy as np
for lvl in (2, 3, 4, 5, 6):
    r = json.load(open(f"{ROOT}/probe/relabel_new_shared_d16_s0.json"))["levels"][f"{lvl}_centered"]
    print(f"level {lvl}: chance={r['chance'][0]:.2f}")
    for k in ("acc_AA", "acc_AA2", "acc_GG"): print(f"  {k:8s}", np.round(r[k], 2).tolist())

level 2: chance=0.53
  acc_AA   [0.53, 0.53, 0.53, 0.53, 0.53, 0.53, 0.53, 0.53, 0.53, 0.53, 0.53, 0.53, 0.62]
  acc_AA2  [0.53, 0.53, 0.53, 0.53, 0.53, 0.53, 0.53, 0.53, 0.53, 0.53, 0.53, 0.53, 0.62]
  acc_GG   [0.53, 0.53, 0.54, 0.53, 0.53, 0.53, 0.53, 0.53, 0.53, 0.53, 0.53, 0.53, 0.56]
level 3: chance=0.33
  acc_AA   [0.33, 0.33, 0.33, 0.33, 0.33, 0.33, 0.33, 0.34, 0.34, 0.33, 0.33, 0.33, 0.35]
  acc_AA2  [0.33, 0.33, 0.33, 0.33, 0.33, 0.33, 0.33, 0.34, 0.34, 0.33, 0.33, 0.33, 0.35]
  acc_GG   [0.33, 0.34, 0.33, 0.33, 0.33, 0.33, 0.33, 0.33, 0.33, 0.33, 0.33, 0.33, 0.33]
level 4: chance=0.37
  acc_AA   [0.37, 0.37, 0.37, 0.37, 0.37, 0.37, 0.37, 0.38, 0.38, 0.39, 0.38, 0.37, 0.38]
  acc_AA2  [0.37, 0.37, 0.37, 0.37, 0.37, 0.37, 0.37, 0.38, 0.38, 0.38, 0.38, 0.37, 0.38]
  acc_GG   [0.37, 0.38, 0.37, 0.37, 0.37, 0.37, 0.38, 0.37, 0.37, 0.37, 0.37, 0.37, 0.37]
level 5: chance=0.37
  acc_AA   [0.39, 0.42, 0.43, 0.44, 0.45, 0.44, 0.47, 0.49, 0.48, 0.47, 0.46, 0.44, 0.43]
  acc_AA2  [0.39

In [13]:
import json, numpy as np
print("max over layers of (acc_AA - chance), shared models, per rendering")
gap = {}
for lvl in (2, 3, 4, 5, 6):
    row = {}
    for g in X.G_LIST:
        p = f"{ROOT}/probe/{g}_shared_d16_s0.json"
        if os.path.exists(p):
            r = json.load(open(p))["levels"][f"{lvl}_centered"]; row[g] = float(np.max(np.array(r["acc_AA"]) - r["chance"][0]))
    gap[lvl] = row; print(f"level {lvl}: n={len(row)}  " + "  ".join(f"{g}={v:.2f}" for g, v in row.items()))
ok_lvls = [l for l in (2, 3, 4, 5, 6) if sum(v >= 0.10 for v in gap[l].values()) >= 8]
print("\nlevels where A decodes by >= 0.10 in >= 8 renderings:", ok_lvls, "| rule picks:", ok_lvls[0] if ok_lvls else "NONE")

max over layers of (acc_AA - chance), shared models, per rendering
level 2: n=4  relabel_new=0.09  relabel_perm=0.13  marker=0.06  swap5=0.26
level 3: n=4  relabel_new=0.03  relabel_perm=0.02  marker=0.03  swap5=0.09
level 4: n=4  relabel_new=0.01  relabel_perm=0.01  marker=0.01  swap5=0.01
level 5: n=4  relabel_new=0.12  relabel_perm=0.11  marker=0.12  swap5=0.09
level 6: n=4  relabel_new=0.27  relabel_perm=0.28  marker=0.27  swap5=0.27

levels where A decodes by >= 0.10 in >= 8 renderings: [] | rule picks: NONE


In [14]:
print(sorted(f for f in os.listdir(f"{ROOT}/probe") if "_shared_" in f))
print("exp8_lib processes:", n_running(), "| probe files total:", len(os.listdir(f"{ROOT}/probe")))

['marker_shared_d16_s0.json', 'relabel_new_shared_d16_s0.json', 'relabel_perm_shared_d16_s0.json', 'swap5_shared_d16_s0.json']
exp8_lib processes: 0 | probe files total: 11


In [ ]:
for g in X.G_LIST:
    for cond, untr in (("shared", False), ("shared", True)):
        try: run_probe(ROOT, g, cond, 16, 0, DEVICE, untrained=untr)
        except Exception as e: print("FAILED", g, cond, "untrained" if untr else "", repr(e)[:300])

In [16]:
# 7c. read the probe result (CPU). Costs from the long-run translators if they exist, else the 24k ones (say which in the report).
cost_root = ROOT_LONG if all(os.path.exists(X.tr_paths(ROOT_LONG, g, 16, 0)) for g in ["identity"] + X.G_LIST) else ROOT
probe_summary(ROOT, d=16, seed=0, cost_root=cost_root)
# Diagnostic for the disjoint column: are nonterminal label spaces the same across A and G in the disjoint condition? If the rules differ for the same symbol, A->G transfer there is near chance by construction.
try:
    for cond in ("shared", "disjoint"):
        gr = X.make_grammars(cond)
        for lg in ("A", "A2", "G"): print(cond, lg, "rules for symbol (2, 0):", gr[lg][(2, 0)])
except Exception as e: print("diagnostic skipped:", e)

=== PROBE: transfer of an A-trained level-4 probe to rendering g (d=16, seed 0; costs from ./exp8) ===
g                a(g)  meanT shared  meanT untrained      dT  #layers T defined  g->g acc (last)  meanT disjoint*
* disjoint column is descriptive only: its label space may not align with A's (see STEP 7c).

VERDICT: UNTESTABLE / NO VERDICT YET - 0 renderings with a defined dT, need 6.
shared A rules for symbol (2, 0): [((3, 2), (3, 1)), ((3, 0), (3, 0), (3, 2))]
shared A2 rules for symbol (2, 0): [((3, 2), (3, 1)), ((3, 0), (3, 0), (3, 2))]
shared G rules for symbol (2, 0): [((3, 2), (3, 1)), ((3, 0), (3, 0), (3, 2))]
disjoint A rules for symbol (2, 0): [((3, 3), (3, 0), (3, 2)), ((3, 0), (3, 2), (3, 1))]
disjoint A2 rules for symbol (2, 0): [((3, 3), (3, 0), (3, 2)), ((3, 0), (3, 2), (3, 1))]
disjoint G rules for symbol (2, 0): [((3, 2), (3, 1)), ((3, 0), (3, 0), (3, 2))]


In [92]:
for g in ["rev4", "rev45", "perm_sym"]:
    j = json.load(open(f"./exp8_12long/translator/{g}_d12_s0.json"))
    c = j["curve"]
    print(g, "last 6 curve points:", [round(x,4) for x in c[-6:]])

rev4 last 6 curve points: [0.4574, 0.4552, 0.4556, 0.455, 0.4528, 0.4558]
rev45 last 6 curve points: [0.4118, 0.4122, 0.412, 0.4115, 0.41, 0.4108]
perm_sym last 6 curve points: [0.4589, 0.4553, 0.4555, 0.4555, 0.4571, 0.456]


## STEP 8. Phase E: seeds 7 and 13 at d=12, then Check 1 and Check 2 (only after the d=12 gate PASSES on seed 0)

In [17]:
# 8a. launch seeds 7 and 13 for D12_SET
if not LAUNCH_D12_SEEDS: print("skipped (flag False)")
else:
    ok = translator_validity(ROOT, D_NEW, verbose=False)
    if not ok: print("V1 not passed at d=12: not launching.")
    else:
        lines = [l for s in (7, 13) for l in joint_lines(ROOT, D_NEW, s, D12_SET, ("shared", "disjoint"), DEVICE)]
        launch(lines, "jt_d12_seeds", ROOT)

skipped (flag False)


In [18]:
# 8b. complete 3-seed read. Also needs translators at seeds 7 and 13 for Check 2 (launch them like 4a with --seed 7 / 13 if you want it).
if read_gate(ROOT, D_NEW, SEEDS, D12_SET):
    r1 = check1(ROOT, d=D_NEW, seeds=SEEDS)
    r2 = check2(ROOT, d=D_NEW, seeds=SEEDS)

d=12: 0 of 24 joint JSONs present | exp8_lib processes running: 0
  missing: [('marker', 'shared', 0), ('marker', 'shared', 7), ('marker', 'shared', 13), ('marker', 'disjoint', 0), ('marker', 'disjoint', 7), ('marker', 'disjoint', 13), ('rev4', 'shared', 0), ('rev4', 'shared', 7), ('rev4', 'shared', 13), ('rev4', 'disjoint', 0), ('rev4', 'disjoint', 7), ('rev4', 'disjoint', 13), ('rev45', 'shared', 0), ('rev45', 'shared', 7), ('rev45', 'shared', 13), ('rev45', 'disjoint', 0), ('rev45', 'disjoint', 7), ('rev45', 'disjoint', 13), ('rev_all', 'shared', 0), ('rev_all', 'shared', 7), ('rev_all', 'shared', 13), ('rev_all', 'disjoint', 0), ('rev_all', 'disjoint', 7), ('rev_all', 'disjoint', 13)]
NOT COMPLETE: no gate read. Re-run this cell when every file is present.


## How to read the outcomes (written before the data)

- **V1 fails at d=12:** the translator needs a width between 12 and 16. Try d=14 once, not a sweep. If nothing passes V1 and V2, the finding is that no tested capacity supports the loss-based test. That goes to Anwoy as is.
- **Check 1 PASS on 3 seeds at d=12:** strongest support available (rev_all shares better than rev4/rev45, which string distance predicts the opposite of).
- **Check 1 FAIL at d=12:** the failure at d=8 was not a capacity artifact. a(g) loses its main argument.
- **Probe PASS:** consistent with the hypothesis through a second instrument. Probe FAIL with Check 1 PASS: the two measures disagree, report both.
- Report every result, failures included. Do not add renderings or thresholds after reading.

In [8]:
print("jobs file exists:", os.path.exists(f"{ROOT}/tr_d12_jobs.sh"))
print(os.popen(f"ls {ROOT}/logs | grep d12 | head -3; tail -n 5 {ROOT}/logs/tr_identity_d12_s0.txt 2>&1").read())

jobs file exists: True
tr_identity_d12_s0.txt
tr_marker_d12_s0.txt
tr_perm_sym_d12_s0.txt
  File "/home/anwoy/exp8_lib.py", line 235, in train_translator
    torch.manual_seed(seed); model = TinyRotaryGPT(VOCAB, d, n_layer, n_head, row_len + 1).to(dev)
  File "/home/anwoy/exp8_lib.py", line 184, in __init__
    super().__init__(); assert n_embd % n_head == 0 and (n_embd // n_head) % 2 == 0, "head_dim must be even for RoPE"
AssertionError: head_dim must be even for RoPE



In [9]:
print(os.popen("grep -n 'add_argument' exp8_lib.py").read())

263:            p = sub.add_parser(name); p.add_argument("--g", required=True, choices=list(TRANSFORMS)); p.add_argument("--seed", type=int, default=0)
264:            p.add_argument("--d", type=int, default=32); p.add_argument("--steps", type=int, default=90000 if name == "joint" else 24000)
265:            p.add_argument("--root", default="./exp8"); p.add_argument("--device", default="cuda:0"); p.add_argument("--n_head", type=int, default=4)
266:            if name == "joint": p.add_argument("--cond", choices=["shared", "disjoint"], required=True); p.add_argument("--big_n", type=int, default=500_000)

